In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

project_dir = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild"
)

print("Project folder exists:", project_dir.exists())

if project_dir.exists():
    print("\nAvailable files:")
    for path in sorted(project_dir.iterdir()):
        if path.is_file():
            print(f"{path.name}  ({path.stat().st_size / 1024:.1f} KB)")

In [ ]:
import pandas as pd
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

# Use the latest saved correction file currently available
tracks_path = project_dir / "all_views_tracks_manual_corrections_v5.csv"
if not tracks_path.exists():
    tracks_path = project_dir / "all_views_botsort_tracks.csv"

tracks = pd.read_csv(tracks_path)

print("TRACKING DATA")
print("File:", tracks_path.name)
print("Rows:", len(tracks))
print("Columns:", tracks.columns.tolist())

print("\nVIEW SUMMARY")
summary = tracks.groupby("view_id").agg(
    detections=("frame_index", "size"),
    unique_original_tracks=("track_id", "nunique"),
    first_frame=("frame_index", "min"),
    last_frame=("frame_index", "max")
)
display(summary)

print("\nMISSING VALUES")
display(tracks.isna().sum().to_frame("missing_values"))

print("\nVIDEO CHECK")
video_candidates = [
    Path("/content/combined_cctv.mp4"),
    project_dir / "combined_cctv.mp4"
]
for path in video_candidates:
    print(f"{path}: {'FOUND' if path.exists() else 'not found'}")

print("\nZONE ANNOTATION FILES")
for path in project_dir.glob("*zone*.json"):
    print(path.name)

In [ ]:
import json
import pandas as pd
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")
tracks_path = project_dir / "all_views_tracks_manual_corrections_v5.csv"
zones_path = Path("/content/combined_cctv__2__zones.json")

with open(zones_path, "r") as f:
    zone_data = json.load(f)

tracks = pd.read_csv(tracks_path)

print("PROJECT:", zone_data.get("project"))
print("VIDEO RESOLUTION:", zone_data.get("video_resolution"))
print("FPS:", zone_data.get("fps"))
print("VIEW DURATION:", zone_data.get("view_duration_sec"), "seconds")
print("\nZONE SUMMARY")

zone_rows = []
for view in zone_data["views"]:
    view_id = int(view["view_id"])
    view_zones = view.get("zones", [])

    for zone in view_zones:
        points = zone.get("points", [])
        zone_rows.append({
            "view_id": view_id,
            "zone_name": zone.get("name"),
            "zone_type": zone.get("type"),
            "polygon_points": len(points),
            "tracking_rows_in_view": int((tracks["view_id"] == view_id).sum()),
            "start_sec": view.get("start_sec"),
            "end_sec": view.get("end_sec")
        })

zone_summary = pd.DataFrame(zone_rows)
display(zone_summary)

print("\nVIEWS PRESENT IN TRACKS:", sorted(tracks["view_id"].unique().tolist()))
print("VIEWS PRESENT IN JSON:", sorted(int(v["view_id"]) for v in zone_data["views"]))
print("TOTAL ZONES:", len(zone_rows))

In [ ]:
import json
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")
tracks_path = project_dir / "all_views_tracks_manual_corrections_v5.csv"
zones_path = Path("/content/combined_cctv__2__zones.json")

tracks = pd.read_csv(tracks_path)
with open(zones_path, "r") as f:
    zone_data = json.load(f)

FPS = float(zone_data["fps"])
GAP_TOLERANCE_FRAMES = 2  # bridge only brief gaps of up to 2 frames

# Build a zone membership table using the foot point (foot_x, foot_y).
membership_rows = []

for view in zone_data["views"]:
    view_id = int(view["view_id"])

    view_tracks = tracks[tracks["view_id"] == view_id].copy()
    view_tracks["track_key"] = view_tracks["corrected_track_id"].astype(str)

    for zone in view["zones"]:
        polygon = np.array(zone["points"], dtype=np.int32).reshape((-1, 1, 2))
        zone_name = zone["name"]

        inside = view_tracks.apply(
            lambda r: cv2.pointPolygonTest(
                polygon,
                (float(r["foot_x"]), float(r["foot_y"])),
                False
            ) >= 0,
            axis=1
        )

        selected = view_tracks.loc[inside, [
            "frame_index", "timestamp_sec", "view_id",
            "track_id", "corrected_track_id", "track_key"
        ]].copy()

        selected["zone_name"] = zone_name
        membership_rows.append(selected)

membership = (
    pd.concat(membership_rows, ignore_index=True)
    if membership_rows else pd.DataFrame()
)

# Count contiguous observed intervals as visits.
visit_rows = []

if not membership.empty:
    membership = membership.sort_values(
        ["view_id", "zone_name", "track_key", "frame_index"]
    )

    group_cols = ["view_id", "zone_name", "track_key"]

    for (view_id, zone_name, track_key), group in membership.groupby(group_cols):
        frames = np.sort(group["frame_index"].unique())

        if len(frames) == 0:
            continue

        # Split visits when the frame gap exceeds the allowed tolerance.
        split_points = np.where(np.diff(frames) > GAP_TOLERANCE_FRAMES + 1)[0] + 1
        frame_segments = np.split(frames, split_points)

        for segment in frame_segments:
            if len(segment) == 0:
                continue

            # Use observed frames only; do not fill gaps into dwell duration.
            dwell_sec = len(segment) / FPS

            visit_rows.append({
                "view_id": int(view_id),
                "zone_name": zone_name,
                "track_key": track_key,
                "start_frame": int(segment[0]),
                "end_frame": int(segment[-1]),
                "observed_frames": int(len(segment)),
                "observed_dwell_sec": round(dwell_sec, 3)
            })

visits = pd.DataFrame(visit_rows)

# Include zones with zero observed visits.
all_zones = [
    (int(view["view_id"]), zone["name"], zone.get("type", "unknown"))
    for view in zone_data["views"]
    for zone in view["zones"]
]

summary_rows = []
for view_id, zone_name, zone_type in all_zones:
    z = visits[
        (visits["view_id"] == view_id) &
        (visits["zone_name"] == zone_name)
    ] if not visits.empty else pd.DataFrame()

    dwell_by_track = z.groupby("track_key")["observed_dwell_sec"].sum() if not z.empty else pd.Series(dtype=float)

    summary_rows.append({
        "view_id": view_id,
        "zone_name": zone_name,
        "zone_type": zone_type,
        "visits": len(z),
        "unique_tracks": int(z["track_key"].nunique()) if not z.empty else 0,
        "total_observed_dwell_sec": round(z["observed_dwell_sec"].sum(), 2) if not z.empty else 0.0,
        "mean_visit_dwell_sec": round(z["observed_dwell_sec"].mean(), 2) if not z.empty else 0.0,
        "max_visit_dwell_sec": round(z["observed_dwell_sec"].max(), 2) if not z.empty else 0.0,
        "mean_dwell_per_track_sec": round(dwell_by_track.mean(), 2) if len(dwell_by_track) else 0.0
    })

zone_dwell_summary = pd.DataFrame(summary_rows)

print("ZONE DWELL SUMMARY")
display(zone_dwell_summary)

print("\nVISIT RECORDS (first 20)")
display(visits.head(20))

# Save reproducible outputs for the submission notebook.
output_dir = Path("/content/vendora_analytics_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

zone_dwell_summary.to_csv(output_dir / "zone_dwell_summary.csv", index=False)
visits.to_csv(output_dir / "zone_visits.csv", index=False)
membership.to_csv(output_dir / "zone_membership.csv", index=False)

print("\nSaved outputs to:", output_dir)

In [ ]:
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

tracks_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)
zones_path = Path("/content/combined_cctv__2__zones.json")
output_dir = Path("/content/vendora_analytics_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

tracks = pd.read_csv(tracks_path)
with open(zones_path, "r") as f:
    zone_data = json.load(f)

W = int(zone_data["video_resolution"]["width"])
H = int(zone_data["video_resolution"]["height"])

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
heatmap_rows = []

for view, ax in zip(zone_data["views"], axes.flat):
    view_id = int(view["view_id"])
    data = tracks[tracks["view_id"] == view_id]

    # Pixel-level foot-position frequency, displayed as a smoothed-looking
    # coarse grid rather than pretending each pixel is independently meaningful.
    heat, xedges, yedges = np.histogram2d(
        data["foot_x"],
        data["foot_y"],
        bins=[64, 36],
        range=[[0, W], [0, H]]
    )

    image = ax.imshow(
        heat.T,
        origin="upper",
        extent=[0, W, H, 0],
        aspect="auto",
        interpolation="bilinear"
    )

    for zone in view.get("zones", []):
        points = np.array(zone["points"], dtype=np.int32)
        if len(points) > 1 and np.array_equal(points[0], points[-1]):
            points = points[:-1]

        polygon = plt.Polygon(
            points,
            closed=True,
            fill=False,
            linewidth=1.5
        )
        ax.add_patch(polygon)

        # Label near the polygon's average point
        center_x, center_y = points.mean(axis=0)
        ax.text(center_x, center_y, zone["name"],
                fontsize=8, ha="center", va="center")

    ax.set_title(f"View {view_id} — {len(data)} detections")
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.set_xlabel("X position (pixels)")
    ax.set_ylabel("Y position (pixels)")

    heatmap_rows.append({
        "view_id": view_id,
        "detections": len(data),
        "occupied_heatmap_bins": int((heat > 0).sum()),
        "peak_bin_detections": int(heat.max()) if heat.size else 0
    })

fig.colorbar(image, ax=list(axes.flat), label="Detection count per spatial bin",
             shrink=0.8)
fig.suptitle("Vendora — Foot-Position Density by Camera View", fontsize=15)
fig.tight_layout()
fig.savefig(output_dir / "six_view_foot_traffic_heatmaps.png",
            dpi=160, bbox_inches="tight")
plt.show()

heatmap_summary = pd.DataFrame(heatmap_rows)
display(heatmap_summary)

heatmap_summary.to_csv(
    output_dir / "heatmap_summary.csv", index=False
)

print("Saved heatmaps and summary to:", output_dir)

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")
tracks = pd.read_csv(project_dir / "all_views_tracks_manual_corrections_v5.csv")

with open("/content/combined_cctv__2__zones.json", "r") as f:
    zone_data = json.load(f)

FPS = float(zone_data["fps"])
output_dir = Path("/content/vendora_analytics_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Calculate movement only between consecutive observations of the same
# corrected track in the same view. Ignore long gaps.
track_data = tracks.sort_values(
    ["view_id", "corrected_track_id", "frame_index"]
).copy()

track_data["previous_frame"] = track_data.groupby(
    ["view_id", "corrected_track_id"]
)["frame_index"].shift()

track_data["previous_x"] = track_data.groupby(
    ["view_id", "corrected_track_id"]
)["foot_x"].shift()

track_data["previous_y"] = track_data.groupby(
    ["view_id", "corrected_track_id"]
)["foot_y"].shift()

track_data["frame_gap"] = (
    track_data["frame_index"] - track_data["previous_frame"]
)

track_data["pixel_distance"] = np.hypot(
    track_data["foot_x"] - track_data["previous_x"],
    track_data["foot_y"] - track_data["previous_y"]
)

# Pixel speed is not physical walking speed; camera calibration is unavailable.
valid = (
    (track_data["frame_gap"] > 0) &
    (track_data["frame_gap"] <= 3)
)

track_data["pixel_speed_per_sec"] = np.where(
    valid,
    track_data["pixel_distance"] * FPS / track_data["frame_gap"],
    np.nan
)

# Per-frame occupancy and speed summaries.
frame_stats = track_data.groupby(
    ["view_id", "frame_index"]
).agg(
    people_observed=("corrected_track_id", "nunique"),
    median_pixel_speed=("pixel_speed_per_sec", "median")
).reset_index()

occupancy_summary = frame_stats.groupby("view_id").agg(
    mean_people_per_frame=("people_observed", "mean"),
    peak_people_per_frame=("people_observed", "max"),
    frames_with_multiple_people=(
        "people_observed", lambda x: int((x >= 3).sum())
    ),
    median_pixel_speed_per_sec=("median_pixel_speed", "median")
).reset_index()

occupancy_summary["mean_people_per_frame"] = (
    occupancy_summary["mean_people_per_frame"].round(2)
)
occupancy_summary["median_pixel_speed_per_sec"] = (
    occupancy_summary["median_pixel_speed_per_sec"].round(2)
)

print("VIEW-LEVEL OCCUPANCY AND MOVEMENT")
display(occupancy_summary)

# Zone-level presence per frame, useful for comparing busy and quiet zones.
zone_presence_rows = []

for view in zone_data["views"]:
    view_id = int(view["view_id"])
    view_tracks = tracks[tracks["view_id"] == view_id]

    for zone in view.get("zones", []):
        pts = np.asarray(zone["points"], dtype=np.int32).reshape((-1, 1, 2))

        inside = view_tracks.apply(
            lambda r: cv2.pointPolygonTest(
                pts, (float(r["foot_x"]), float(r["foot_y"])), False
            ) >= 0,
            axis=1
        )

        z = view_tracks.loc[inside]
        people_by_frame = z.groupby("frame_index")[
            "corrected_track_id"
        ].nunique()

        # Include frames with zero people in the zone.
        start_frame = int(round(float(view["start_sec"]) * FPS))
        end_frame = int(round(float(view["end_sec"]) * FPS))
        all_frames = pd.RangeIndex(start_frame, end_frame)

        people_by_frame = people_by_frame.reindex(
            all_frames, fill_value=0
        )

        zone_presence_rows.append({
            "view_id": view_id,
            "zone_name": zone["name"],
            "zone_type": zone.get("type", "unknown"),
            "mean_people_per_frame": round(float(people_by_frame.mean()), 3),
            "peak_people_in_frame": int(people_by_frame.max()),
            "frames_with_anyone": int((people_by_frame > 0).sum()),
            "frames_with_3plus_people": int((people_by_frame >= 3).sum()),
            "observation_window_sec": round(len(all_frames) / FPS, 2)
        })

zone_presence = pd.DataFrame(zone_presence_rows)

print("\nZONE-LEVEL PRESENCE")
display(zone_presence)

occupancy_summary.to_csv(
    output_dir / "view_occupancy_movement_summary.csv", index=False
)
zone_presence.to_csv(
    output_dir / "zone_occupancy_summary.csv", index=False
)

print("\nSaved occupancy and movement outputs to:", output_dir)

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path("/content/vendora_analytics_outputs")

flags = zone_presence.copy()

# Normalize activity against each zone's 10-second observation window.
flags["presence_ratio"] = (
    flags["frames_with_anyone"] /
    (flags["observation_window_sec"] * FPS)
)

flags["crowded_frame_ratio"] = (
    flags["frames_with_3plus_people"] /
    (flags["observation_window_sec"] * FPS)
)

# Relative thresholds across the annotated zones.
high_occupancy_threshold = flags["mean_people_per_frame"].quantile(0.75)
low_occupancy_threshold = flags["mean_people_per_frame"].quantile(0.25)

def classify_zone(row):
    high_occupancy = (
        row["mean_people_per_frame"] >= high_occupancy_threshold
    )
    repeated_crowding = row["crowded_frame_ratio"] >= 0.75

    low_activity = (
        row["mean_people_per_frame"] <= low_occupancy_threshold
        and row["presence_ratio"] <= 0.75
    )

    if high_occupancy and repeated_crowding:
        return "Bottleneck candidate — review video"
    elif low_activity:
        return "Low-activity candidate — review video"
    else:
        return "No preliminary flag"

flags["preliminary_flag"] = flags.apply(classify_zone, axis=1)

print("Rule thresholds:")
print(f"High occupancy: >= {high_occupancy_threshold:.3f} people/frame")
print(f"Low occupancy: <= {low_occupancy_threshold:.3f} people/frame")
print("\nPRELIMINARY ZONE FLAGS")

display(flags[[
    "view_id",
    "zone_name",
    "mean_people_per_frame",
    "peak_people_in_frame",
    "presence_ratio",
    "crowded_frame_ratio",
    "preliminary_flag"
]])

flags.to_csv(
    output_dir / "preliminary_zone_flags.csv",
    index=False
)

print("\nSaved to:", output_dir / "preliminary_zone_flags.csv")

In [ ]:
from pathlib import Path
import zipfile

output_dir = Path("/content/vendora_analytics_outputs")
zip_path = Path("/content/vendora_submission_outputs.zip")

files_to_include = [
    "README.md",
    "requirements.txt",
    "zone_dwell_summary.csv",
    "zone_visits.csv",
    "zone_membership.csv",
    "six_view_foot_traffic_heatmaps.png",
    "heatmap_summary.csv",
    "view_occupancy_movement_summary.csv",
    "zone_occupancy_summary.csv",
    "preliminary_zone_flags.csv",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in files_to_include:
        path = output_dir / filename
        if path.exists():
            zf.write(path, arcname=filename)
        else:
            print("Missing:", filename)

print("ZIP created:", zip_path)
print("Size:", round(zip_path.stat().st_size / 1024, 1), "KB")

In [ ]:
from google.colab import files
files.download("/content/vendora_submission_outputs.zip")

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/vendora_botsort_reid_rebuild')

# 1. Inspect the latest tracking CSV
print("=" * 70)
print("1. FINAL TRACKING CSV — v5")
print("=" * 70)

tracks = pd.read_csv(BASE / 'all_views_tracks_manual_corrections_v5.csv')
print("Shape:", tracks.shape)
print("Columns:", tracks.columns.tolist())
print(tracks.head(3).to_string(index=False))

# 2. Inspect the ReID embedding files
print("\n" + "=" * 70)
print("2. REID EMBEDDINGS")
print("=" * 70)

for filename in ['reid_crop_embeddings.npy', 'reid_track_embeddings.npy']:
    arr = np.load(BASE / filename, mmap_mode='r')
    print(f"{filename}: shape={arr.shape}, dtype={arr.dtype}")

# 3. Inspect the metadata that maps embeddings to crops/tracks
for filename in [
    'reid_crop_metadata.csv',
    'reid_track_embedding_index.csv',
    'reid_cross_view_similarity_pairs.csv',
    'reviewed_global_identity_mapping.csv',
    'global_identity_validation_summary.csv',
    'all_views_track_quality_summary.csv'
]:
    path = BASE / filename
    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)
    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

# 4. Inspect BoT-SORT ReID configuration
print("\n" + "=" * 70)
print("4. BOTSORT REID CONFIGURATION")
print("=" * 70)

for filename in ['botsort_reid.yaml', 'run_metadata.yaml']:
    path = BASE / filename
    print(f"\n--- {filename} ---")
    print(path.read_text())

# 5. Check corrected-ID coverage
print("\n" + "=" * 70)
print("5. TRACKING ID / CORRECTION COVERAGE")
print("=" * 70)

for col in ['view_id', 'track_id', 'corrected_track_id']:
    if col in tracks.columns:
        print(f"{col}: unique={tracks[col].nunique()}, missing={tracks[col].isna().sum()}")

if 'corrected_track_id' in tracks.columns and 'track_id' in tracks.columns:
    changed = (
        tracks['corrected_track_id'].astype(str)
        != tracks['track_id'].astype(str)
    )
    print("Rows where corrected ID differs from original ID:", int(changed.sum()))

print("\nInspection complete. No files were modified.")


In [ ]:

from pathlib import Path
import pandas as pd

BASE = Path('/content/drive/MyDrive/vendora_botsort_reid_rebuild')

files = [
    'candidate_identity_mapping.csv',
    'provisional_global_identity_mapping.csv',
    'reviewed_global_identity_mapping.csv',
    'global_identity_frame_collisions.csv',
    'global_identity_track_continuity.csv',
    'same_frame_group_conflict_audit.csv',
    'duplicate_review_decisions.csv',
    'reid_corrected_human_review.csv',
]

for name in files:
    path = BASE / name
    print('\n' + '=' * 75)
    print(name)
    print('=' * 75)

    if not path.exists():
        print('FILE MISSING')
        continue

    df = pd.read_csv(path)
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    print(df.head(8).to_string(index=False))

# Check how much of the video the saved final tracking CSV covers
tracks = pd.read_csv(BASE / 'all_views_tracks_manual_corrections_v5.csv')

print('\n' + '=' * 75)
print('FINAL TRACKING COVERAGE')
print('=' * 75)
print('Frame range:', tracks.frame_index.min(), 'to', tracks.frame_index.max())
print('Views:', sorted(tracks.view_id.unique().tolist()))
print('Rows per view:')
print(tracks.groupby('view_id').size().to_string())

print('\nSaved run metadata:')
print((BASE / 'run_metadata.yaml').read_text())


In [ ]:
!pip install ultralytics

In [ ]:

from ultralytics import YOLO
from pathlib import Path

model = YOLO("yolov8n.pt")
model_path = Path(model.ckpt_path) if getattr(model, "ckpt_path", None) else Path("/content/yolov8n.pt")

print("Model loaded successfully.")
print("Model path:", model_path.resolve())
print("Model file exists:", model_path.exists())


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import cv2
import torch

video_path = Path("/content/combined_cctv.mp4")
model_path = Path("/content/yolov8n.pt")
tracker_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/botsort_reid.yaml"
)

assert video_path.exists()
assert model_path.exists()
assert tracker_path.exists()

cap = cv2.VideoCapture(str(video_path))
model = YOLO(str(model_path))

print("Testing first 10 frames individually...")

for frame_index in range(10):
    ok, frame = cap.read()
    if not ok:
        print("Video ended at frame", frame_index)
        break

    result = model.track(
        source=frame,
        persist=(frame_index > 0),
        tracker=str(tracker_path),
        classes=[0],
        conf=0.1,
        iou=0.7,
        verbose=False,
        device=0 if torch.cuda.is_available() else "cpu",
    )[0]

    print(
        f"Frame {frame_index}: "
        f"detections={len(result.boxes) if result.boxes is not None else 0}, "
        f"result_feats={getattr(result, 'feats', None) is not None}"
    )

    if frame_index == 0:
        print("Tracker class:", type(model.predictor.trackers[0]).__name__)
        print("Encoder class:", type(model.predictor.trackers[0].encoder).__name__
              if model.predictor.trackers[0].encoder is not None else None)

cap.release()
print("Diagnostic complete. No project output files were written.")


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import cv2
import torch

video_path = Path("/content/combined_cctv.mp4")
model_path = Path("/content/yolov8n.pt")
tracker_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/botsort_reid.yaml"
)

assert video_path.exists() and model_path.exists() and tracker_path.exists()

cap = cv2.VideoCapture(str(video_path))
model = YOLO(str(model_path))

for frame_index in range(10):
    ok, frame = cap.read()
    if not ok:
        break

    result = model.track(
        source=frame,
        persist=(frame_index > 0),
        tracker=str(tracker_path),
        classes=[0],
        conf=0.1,
        iou=0.7,
        verbose=False,
        device=0 if torch.cuda.is_available() else "cpu",
    )[0]

    predictor = model.predictor
    tracker = predictor.trackers[0]
    feat_maps = getattr(predictor, "_feats", None)

    if feat_maps is None:
        feature_status = "No feature maps retained"
    else:
        feature_status = [
            tuple(x.shape) if hasattr(x, "shape") else type(x).__name__
            for x in feat_maps
        ]

    print(
        f"Frame {frame_index}: "
        f"detections={len(result.boxes)}, "
        f"hook_registered={getattr(predictor, '_hook', None) is not None}, "
        f"feature_maps={feature_status}, "
        f"encoder={type(tracker.encoder).__name__ if tracker.encoder is not None else None}"
    )

cap.release()
print("Finished; no project outputs written.")


In [ ]:

import cv2
import torch
from ultralytics import YOLO
from pathlib import Path

video_path = "/content/combined_cctv.mp4"
model_path = "/content/yolov8n.pt"
tracker_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "botsort_reid.yaml"
)

cap = cv2.VideoCapture(video_path)
model = YOLO(model_path)

original_encoder = None
try:
    for frame_index in range(5):
        ok, frame = cap.read()
        if not ok:
            break

        result = model.track(
            source=frame,
            persist=(frame_index > 0),
            tracker=tracker_path,
            classes=[0],
            conf=0.1,
            iou=0.7,
            verbose=False,
            device=0 if torch.cuda.is_available() else "cpu",
        )[0]

        tracker = model.predictor.trackers[0]

        if frame_index == 0:
            original_encoder = tracker.encoder

            def diagnostic_encoder(features, detections):
                print(
                    "Encoder input:",
                    type(features).__name__,
                    "detections:",
                    len(detections),
                )
                output = original_encoder(features, detections)
                print(
                    "Feature vectors:",
                    len(output),
                    "first shape:",
                    getattr(output[0], "shape", None) if output else None,
                )
                return output

            tracker.encoder = diagnostic_encoder

        print(f"Frame {frame_index}: {len(result.boxes)} returned detections")

finally:
    cap.release()

    # Restore the encoder if the tracker exists.
    if getattr(model, "predictor", None) is not None:
        for tracker in getattr(model.predictor, "trackers", []):
            if original_encoder is not None:
                tracker.encoder = original_encoder

print("Diagnostic complete; no project files written.")


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

files = {
    "mapping": "provisional_global_identity_mapping.csv",
    "review": "reid_corrected_human_review.csv",
    "collisions": "global_identity_frame_collisions.csv",
    "summary": "global_identity_validation_summary.csv",
}

for key, filename in files.items():
    path = base / filename
    print(f"{key}: {'FOUND' if path.exists() else 'MISSING'} — {filename}")

if all((base / files[k]).exists() for k in files):
    mapping = pd.read_csv(base / files["mapping"])
    review = pd.read_csv(base / files["review"])
    collisions = pd.read_csv(base / files["collisions"])
    summary = pd.read_csv(base / files["summary"])

    print("\n=== PROVISIONAL MAPPING ===")
    print("Rows:", len(mapping))
    print("Columns:", mapping.columns.tolist())
    print(mapping.to_string(index=False))

    print("\n=== HUMAN REVIEW LABELS ===")
    print("Columns:", review.columns.tolist())
    for col in review.columns:
        if any(term in col.lower() for term in ["decision", "label", "judgment", "same_person"]):
            print(f"\n{col} counts:")
            print(review[col].value_counts(dropna=False).to_string())

    print("\n=== SAME-FRAME GLOBAL ID COLLISIONS ===")
    print("Rows:", len(collisions))
    print("Columns:", collisions.columns.tolist())
    print(collisions.head(10).to_string(index=False))

    print("\n=== GLOBAL ID SUMMARY ===")
    print(summary.to_string(index=False))


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

mapping = pd.read_csv(base / "provisional_global_identity_mapping.csv")
review = pd.read_csv(base / "reid_corrected_human_review.csv")
tracks_path = base / "all_views_botsort_tracks.csv"

# Map each local track key to its provisional global identity.
track_to_global = dict(zip(mapping["track_key"], mapping["global_person_id"]))

print("=== REVIEWED DIFFERENT-PEOPLE PAIRS MERGED TOGETHER ===")

review = review.copy()
review["global_a"] = review["track_a"].map(track_to_global)
review["global_b"] = review["track_b"].map(track_to_global)

merged_negative = review[
    (review["human_label"] == "different_people")
    & review["global_a"].notna()
    & (review["global_a"] == review["global_b"])
]

print("Conflicting reviewed pairs:", len(merged_negative))
if len(merged_negative):
    print(merged_negative[
        ["track_a", "track_b", "cosine_similarity", "human_label", "global_a"]
    ].to_string(index=False))
else:
    print("No reviewed negative pair directly shares a global ID.")

print("\n=== SAME-FRAME CONFLICTS FROM ORIGINAL TRACKS ===")

tracks = pd.read_csv(tracks_path)
tracks["track_key"] = (
    "V" + tracks["view_id"].astype(str)
    + "_ID" + tracks["track_id"].astype(int).astype(str)
)
tracks["global_person_id"] = tracks["track_key"].map(track_to_global)

mapped = tracks[tracks["global_person_id"].notna()].copy()

conflicts = (
    mapped.groupby(["view_id", "frame_index", "global_person_id"])["track_key"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="local_tracks")
)

conflicts["number_of_tracks"] = conflicts["local_tracks"].map(len)
conflicts = conflicts[conflicts["number_of_tracks"] > 1]

print("Same-frame conflict groups:", len(conflicts))
if len(conflicts):
    print(conflicts.head(30).to_string(index=False))

print("\n=== GLOBAL ID MEMBERS ===")
for global_id, group in mapping.groupby("global_person_id"):
    print(f"{global_id}: {', '.join(group['track_key'].tolist())}")


In [ ]:

from pathlib import Path
import pandas as pd
from itertools import combinations

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

mapping = pd.read_csv(base / "provisional_global_identity_mapping.csv")
review = pd.read_csv(base / "reid_corrected_human_review.csv")
tracks = pd.read_csv(base / "all_views_botsort_tracks.csv")

# Build keys consistently with the existing mapping.
tracks["track_key"] = (
    "V" + tracks["view_id"].astype(str)
    + "_ID" + tracks["track_id"].astype(int).astype(str)
)

# 1. Every explicitly rejected identity pair.
rejected_pairs = set()
for _, row in review[review["human_label"] == "different_people"].iterrows():
    rejected_pairs.add(tuple(sorted([row["track_a"], row["track_b"]])))

print("=== EXPLICITLY REJECTED PAIRS ===")
print("Unique rejected pairs:", len(rejected_pairs))
for a, b in sorted(rejected_pairs):
    print(f"{a} <-> {b}")

# 2. Same-view tracks that overlap in time.
print("\n=== TEMPORAL COEXISTENCE WITHIN EACH PROVISIONAL GROUP ===")

mapped_tracks = tracks.merge(
    mapping[["track_key", "global_person_id"]],
    on="track_key",
    how="inner"
)

for global_id, group in mapping.groupby("global_person_id"):
    keys = group["track_key"].tolist()
    if len(keys) < 2:
        continue

    print(f"\n{global_id}:")
    for a, b in combinations(keys, 2):
        ta = mapped_tracks[mapped_tracks["track_key"] == a]
        tb = mapped_tracks[mapped_tracks["track_key"] == b]

        same_view = (
            not ta.empty and not tb.empty
            and ta["view_id"].iloc[0] == tb["view_id"].iloc[0]
        )

        if same_view:
            fa = set(ta["frame_index"].astype(int))
            fb = set(tb["frame_index"].astype(int))
            overlap = len(fa & fb)
            print(f"  {a} vs {b}: same-view frame overlap = {overlap}")

print("\nAudit complete. No files written.")



In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

tracks_path = base / "all_views_botsort_tracks.csv"
provisional_path = base / "provisional_global_identity_mapping.csv"
output_path = base / "conservative_global_identity_mapping_v1.csv"

tracks = pd.read_csv(tracks_path)
provisional = pd.read_csv(provisional_path)

# Build one unique key per camera-view-local track.
tracks["track_key"] = (
    "V" + tracks["view_id"].astype(int).astype(str)
    + "_ID" + tracks["track_id"].astype(int).astype(str)
)

local_tracks = (
    tracks[["track_key", "view_id", "track_id"]]
    .drop_duplicates()
    .sort_values(["view_id", "track_id"])
    .reset_index(drop=True)
)

# Record the old assignment for auditability.
old_assignments = (
    provisional[["track_key", "global_person_id"]]
    .rename(columns={"global_person_id": "previous_provisional_id"})
)

candidate = local_tracks.merge(
    old_assignments,
    on="track_key",
    how="left"
)

# Do not merge identities across views without new supporting evidence.
candidate["global_person_id"] = (
    "LOCAL_" + candidate["track_key"]
)
candidate["assignment_status"] = "unverified_local_track_only"

# Keep provenance and useful local-track identifiers.
candidate = candidate[
    [
        "global_person_id",
        "track_key",
        "view_id",
        "track_id",
        "previous_provisional_id",
        "assignment_status",
    ]
]

candidate.to_csv(output_path, index=False)

print("Saved new conservative mapping:", output_path)
print("Unique local tracks:", len(candidate))
print("Unique assigned identities:", candidate["global_person_id"].nunique())
print("Tracks with previous provisional assignments:",
      candidate["previous_provisional_id"].notna().sum())
print("\nPreview:")
print(candidate.head(12).to_string(index=False))

assert candidate["global_person_id"].is_unique
assert candidate["track_key"].is_unique
assert output_path.exists()

print("\nValidation passed. Original mapping and tracking files were not overwritten.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

index_path = base / "reid_track_embedding_index.csv"
embeddings_path = base / "reid_track_embeddings.npy"
review_path = base / "reid_corrected_human_review.csv"
tracks_path = base / "all_views_botsort_tracks.csv"

for path in [index_path, embeddings_path, review_path, tracks_path]:
    assert path.exists(), f"Missing required file: {path}"

index = pd.read_csv(index_path)
embeddings = np.load(embeddings_path)
review = pd.read_csv(review_path)
tracks = pd.read_csv(tracks_path)

print("Embedding matrix shape:", embeddings.shape)
print("Index rows:", len(index))
print("Index columns:", index.columns.tolist())
print("Review rows:", len(review))
print("Track rows:", len(tracks))

assert embeddings.ndim == 2
assert len(index) == embeddings.shape[0]
assert index["track_key"].is_unique
assert index["embedding_row"].is_unique
assert index["embedding_row"].min() >= 0
assert index["embedding_row"].max() < len(embeddings)
assert np.isfinite(embeddings).all()

# Verify that each track key follows the expected Vn_IDm convention.
expected_keys = set(
    "V" + tracks["view_id"].astype(int).astype(str)
    + "_ID" + tracks["track_id"].astype(int).astype(str)
)
indexed_keys = set(index["track_key"].astype(str))

print("Unique tracks in tracking CSV:", len(expected_keys))
print("Unique tracks in embedding index:", len(indexed_keys))
print("Embedding tracks absent from tracking CSV:",
      len(indexed_keys - expected_keys))
print("Tracked identities without embeddings:",
      len(expected_keys - indexed_keys))

print("\nHuman-review label counts:")
print(review["human_label"].value_counts(dropna=False).to_string())

print("\nEmbedding index preview:")
print(index.head(10).to_string(index=False))

print("\nValidation passed; no files written.")


In [ ]:

from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

index = pd.read_csv(base / "reid_track_embedding_index.csv")
embeddings = np.load(base / "reid_track_embeddings.npy")
review = pd.read_csv(base / "reid_corrected_human_review.csv")

# Normalize vectors defensively before calculating cosine similarity.
X = embeddings.astype(np.float32)
norms = np.linalg.norm(X, axis=1, keepdims=True)
X = X / np.clip(norms, 1e-12, None)

# Use view-aware track keys from the embedding index.
index = index.reset_index(drop=True)
row_for_track = {
    str(row.track_key): int(row.embedding_row)
    for row in index.itertuples()
}

# Human-reviewed negative pairs are hard exclusions.
rejected_pairs = {
    tuple(sorted((str(row.track_a), str(row.track_b))))
    for row in review.itertuples()
    if row.human_label == "different_people"
}

records = []

for track_a, track_b in combinations(sorted(row_for_track), 2):
    view_a = track_a.split("_")[0]
    view_b = track_b.split("_")[0]

    # Only compare tracks from different camera views.
    if view_a == view_b:
        continue

    ia = row_for_track[track_a]
    ib = row_for_track[track_b]
    similarity = float(np.dot(X[ia], X[ib]))

    pair = tuple(sorted((track_a, track_b)))
    rejected = pair in rejected_pairs

    records.append({
        "track_a": track_a,
        "view_a": view_a,
        "track_b": track_b,
        "view_b": view_b,
        "cosine_similarity": similarity,
        "human_review_status": (
            "rejected_different_people"
            if rejected else "not_yet_reviewed"
        ),
        "eligible_for_review": not rejected,
    })

candidates = pd.DataFrame(records)

if not candidates.empty:
    candidates = candidates.sort_values(
        ["eligible_for_review", "cosine_similarity"],
        ascending=[False, False]
    ).reset_index(drop=True)

output_path = base / "cross_view_reid_candidates_v1.csv"
candidates.to_csv(output_path, index=False)

eligible = candidates[candidates["eligible_for_review"]]
rejected_count = (~candidates["eligible_for_review"]).sum()

print("Saved:", output_path)
print("Total cross-view pairs:", len(candidates))
print("Explicitly rejected pairs excluded from shortlist:", rejected_count)
print("Remaining review candidates:", len(eligible))

print("\nTop 20 eligible candidates:")
print(
    eligible.head(20)[
        ["track_a", "track_b", "cosine_similarity", "human_review_status"]
    ].to_string(index=False)
)

print("\nImportant: similarity is a ranking signal, not proof of identity.")
print("No global identities were merged and no existing files were overwritten.")


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

manifest_path = base / "cross_view_top20_review_manifest.csv"
candidate_path = base / "cross_view_reid_candidates_v1.csv"

assert manifest_path.exists(), f"Missing: {manifest_path}"
assert candidate_path.exists(), f"Missing: {candidate_path}"

manifest = pd.read_csv(manifest_path)
candidates = pd.read_csv(candidate_path)

# Record the user's visual review outcome for these 20 pairs.
manifest["human_decision"] = "different_people"
manifest["review_note"] = (
    "User visually reviewed the pair and confirmed different people."
)

manifest.to_csv(manifest_path, index=False)

reviewed_pairs = manifest[
    ["track_a", "track_b", "cosine_similarity", "human_decision", "review_note"]
].copy()

rejected_path = base / "cross_view_visual_rejections_v1.csv"
reviewed_pairs.to_csv(rejected_path, index=False)

# Mark the corresponding pairs as ineligible in a NEW candidate version.
pair_set = {
    tuple(sorted((str(r.track_a), str(r.track_b))))
    for r in manifest.itertuples(index=False)
}

candidates = candidates.copy()
candidate_pairs = [
    tuple(sorted((str(a), str(b))))
    for a, b in zip(candidates["track_a"], candidates["track_b"])
]

mask = pd.Series(candidate_pairs, index=candidates.index).isin(pair_set)
candidates.loc[mask, "human_review_status"] = "rejected_different_people"
candidates.loc[mask, "eligible_for_review"] = False

updated_candidates_path = base / "cross_view_reid_candidates_v2.csv"
candidates.to_csv(updated_candidates_path, index=False)

print("Recorded visual rejections:", len(manifest))
print("Rejected-pair file:", rejected_path)
print("Updated candidate file:", updated_candidates_path)
print("Remaining eligible candidates:",
      int(candidates["eligible_for_review"].sum()))
print("Original candidate file and all tracking files remain unchanged.")


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")
analytics = Path("/content/vendora_analytics_outputs")

files = {
    "latest_manual_tracks": base / "all_views_tracks_manual_corrections_v5.csv",
    "zone_visits": analytics / "zone_visits.csv",
    "zone_dwell_summary": analytics / "zone_dwell_summary.csv",
    "zone_occupancy_summary": analytics / "zone_occupancy_summary.csv",
    "preliminary_zone_flags": analytics / "preliminary_zone_flags.csv",
    "view_occupancy_movement_summary": analytics / "view_occupancy_movement_summary.csv",
    "heatmap_summary": analytics / "heatmap_summary.csv",
}

for name, path in files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

for name in [
    "zone_visits",
    "zone_dwell_summary",
    "zone_occupancy_summary",
    "preliminary_zone_flags",
    "view_occupancy_movement_summary",
    "heatmap_summary",
]:
    path = files[name]
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n=== {name} | shape={df.shape} ===")
        print("Columns:", df.columns.tolist())
        print(df.head(5).to_string(index=False))

print("\n=== TRACK VERSION CHECK ===")
if files["latest_manual_tracks"].exists():
    tracks = pd.read_csv(files["latest_manual_tracks"])
    print("Rows:", len(tracks))
    print("Columns:", tracks.columns.tolist())
    print("Views:", sorted(tracks["view_id"].unique().tolist()))
    print("Has corrected_track_id:", "corrected_track_id" in tracks.columns)

print("\nRead-only validation complete. No files written.")


In [ ]:
import json
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/vendora_botsort_reid_rebuild")

zone_path = Path("/content/combined_cctv__2__zones.json")
tracks_path = base / "all_views_tracks_manual_corrections_v5.csv"

# Load the original annotations
with open(zone_path, "r", encoding="utf-8") as f:
    zones_data = json.load(f)

# Load corrected tracking data
tracks = pd.read_csv(tracks_path)

print("ZONE JSON")
print("Video:", zones_data.get("video_filename"))
print("Resolution:", zones_data.get("video_resolution"))
print("FPS:", zones_data.get("fps"))
print("Duration:", zones_data.get("duration_sec"))

print("\nZONES BY VIEW")
for view in zones_data["views"]:
    zone_list = view.get("zones", [])
    print(
        f"View {view['view_id']}: "
        f"{view.get('start_sec')}–{view.get('end_sec')} sec | "
        f"{len(zone_list)} zones"
    )
    for zone in zone_list:
        print(f"  - {zone['name']} ({len(zone.get('points', []))} polygon points)")

print("\nTRACKING CSV")
print("Exists:", tracks_path.exists())
print("Rows:", len(tracks))
print("Columns:", tracks.columns.tolist())

if "view_id" in tracks.columns:
    print("\nTracks by view:")
    print(tracks.groupby("view_id").size().to_string())

if "corrected_track_id" in tracks.columns:
    print("\nMissing corrected IDs:", tracks["corrected_track_id"].isna().sum())

In [ ]:

import json
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# ---------- INPUTS ----------
zone_path = Path("/content/combined_cctv__2__zones.json")
tracks_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)
out_dir = Path("/content/vendora_analytics_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

FPS = 24.0
MERGE_GAP_FRAMES = 3
HEATMAP_BIN_PX = 80

# ---------- LOAD DATA ----------
with open(zone_path, "r", encoding="utf-8") as f:
    zone_data = json.load(f)

df = pd.read_csv(tracks_path)
required = {
    "frame_index", "timestamp_sec", "view_id",
    "corrected_track_id", "foot_x", "foot_y"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing tracking columns: {sorted(missing)}")

df = df.dropna(
    subset=["frame_index", "view_id", "corrected_track_id", "foot_x", "foot_y"]
).copy()

df["frame_index"] = df["frame_index"].astype(int)
df["view_id"] = df["view_id"].astype(int)
df["foot_x"] = df["foot_x"].astype(float)
df["foot_y"] = df["foot_y"].astype(float)
df["timestamp_sec"] = pd.to_numeric(df["timestamp_sec"], errors="coerce")

# Use view-scoped identities. Never merge identities across camera views.
df["local_person_key"] = (
    "V" + df["view_id"].astype(str) + "_"
    + df["corrected_track_id"].astype(str)
)

# ---------- POLYGON HELPERS ----------
def normalize_points(points):
    """Accept [[x,y], ...] or [{'x': x, 'y': y}, ...]."""
    normalized = []
    for p in points:
        if isinstance(p, dict):
            x = p.get("x", p.get("X"))
            y = p.get("y", p.get("Y"))
            if x is None or y is None:
                raise ValueError(f"Unrecognized polygon point: {p}")
            normalized.append([float(x), float(y)])
        else:
            normalized.append([float(p[0]), float(p[1])])

    if len(normalized) < 3:
        raise ValueError(f"Polygon needs at least 3 points: {normalized}")
    return np.asarray(normalized, dtype=np.float32)

zone_specs = []
for view in zone_data["views"]:
    for zone in view.get("zones", []):
        poly = normalize_points(zone["points"])
        zone_specs.append({
            "view_id": int(view["view_id"]),
            "view_start_sec": float(view["start_sec"]),
            "view_end_sec": float(view["end_sec"]),
            "zone_name": zone["name"],
            "zone_type": zone.get("type", "unspecified"),
            "polygon": poly,
        })

# ---------- POINT-IN-POLYGON ----------
def inside_polygon(x, y, polygon):
    return cv2.pointPolygonTest(
        polygon, (float(x), float(y)), False
    ) >= 0

visit_rows = []
occupancy_rows = []
zone_detection_rows = []

for z in zone_specs:
    view_id = z["view_id"]
    zone_name = z["zone_name"]
    poly = z["polygon"]

    view_df = df[df["view_id"] == view_id].copy()
    if view_df.empty:
        print(f"WARNING: No tracking rows for view {view_id}")
        continue

    # Test the foot point, not the bounding-box center.
    mask = [
        inside_polygon(x, y, poly)
        for x, y in zip(view_df["foot_x"], view_df["foot_y"])
    ]
    inside = view_df.loc[mask].copy()
    inside["zone_name"] = zone_name
    inside["zone_type"] = z["zone_type"]

    # Per-frame zone occupancy: unique visible local identities.
    if not inside.empty:
        occ = (
            inside.groupby("frame_index")["local_person_key"]
            .nunique()
            .rename("person_count")
            .reset_index()
        )
        for row in occ.itertuples(index=False):
            occupancy_rows.append({
                "view_id": view_id,
                "zone_name": zone_name,
                "frame_index": int(row.frame_index),
                "timestamp_sec": float(row.frame_index / FPS),
                "person_count": int(row.person_count),
            })

    # Record zone detections for auditing.
    for row in inside.itertuples(index=False):
        zone_detection_rows.append({
            "view_id": view_id,
            "zone_name": zone_name,
            "zone_type": z["zone_type"],
            "frame_index": int(row.frame_index),
            "timestamp_sec": float(row.timestamp_sec)
                if pd.notna(row.timestamp_sec) else float(row.frame_index / FPS),
            "local_person_key": row.local_person_key,
            "foot_x": float(row.foot_x),
            "foot_y": float(row.foot_y),
        })

    # Build visits per person and zone; merge short gaps up to 3 frames.
    for person_key, person_rows in inside.groupby("local_person_key"):
        frames = sorted(person_rows["frame_index"].unique().astype(int))
        if not frames:
            continue

        start = prev = frames[0]
        for frame in frames[1:]:
            if frame - prev <= MERGE_GAP_FRAMES + 1:
                prev = frame
            else:
                visit_rows.append({
                    "view_id": view_id,
                    "zone_name": zone_name,
                    "zone_type": z["zone_type"],
                    "local_person_key": person_key,
                    "start_frame": start,
                    "end_frame": prev,
                    "start_sec": start / FPS,
                    "end_sec": prev / FPS,
                    "observed_frames": int(
                        person_rows["frame_index"].between(start, prev).sum()
                    ),
                    "duration_sec": (prev - start + 1) / FPS,
                })
                start = prev = frame

        visit_rows.append({
            "view_id": view_id,
            "zone_name": zone_name,
            "zone_type": z["zone_type"],
            "local_person_key": person_key,
            "start_frame": start,
            "end_frame": prev,
            "start_sec": start / FPS,
            "end_sec": prev / FPS,
            "observed_frames": int(
                person_rows["frame_index"].between(start, prev).sum()
            ),
            "duration_sec": (prev - start + 1) / FPS,
        })

# ---------- OUTPUT 1: ZONE VISITS ----------
visits = pd.DataFrame(visit_rows)
if visits.empty:
    visits = pd.DataFrame(columns=[
        "view_id", "zone_name", "zone_type", "local_person_key",
        "start_frame", "end_frame", "start_sec", "end_sec",
        "observed_frames", "duration_sec"
    ])
visits.to_csv(out_dir / "zone_visits.csv", index=False)

# ---------- OUTPUT 2: DWELL SUMMARY ----------
if not visits.empty:
    dwell = (
        visits.groupby(["view_id", "zone_name", "zone_type"])
        .agg(
            visits=("duration_sec", "size"),
            unique_tracks=("local_person_key", "nunique"),
            total_observed_visit_sec=("duration_sec", "sum"),
            mean_visit_sec=("duration_sec", "mean"),
            median_visit_sec=("duration_sec", "median"),
            max_visit_sec=("duration_sec", "max"),
        ).reset_index()
    )
else:
    dwell = pd.DataFrame(columns=[
        "view_id", "zone_name", "zone_type", "visits", "unique_tracks",
        "total_observed_visit_sec", "mean_visit_sec",
        "median_visit_sec", "max_visit_sec"
    ])
dwell.to_csv(out_dir / "zone_dwell_summary.csv", index=False)

# ---------- OUTPUT 3: ZONE OCCUPANCY SUMMARY ----------
occupancy = pd.DataFrame(occupancy_rows)
if not occupancy.empty:
    zone_occ = (
        occupancy.groupby(["view_id", "zone_name"])
        .agg(
            frames_with_people=("frame_index", "nunique"),
            mean_people_when_occupied=("person_count", "mean"),
            peak_people=("person_count", "max"),
            total_person_frame_observations=("person_count", "sum"),
        ).reset_index()
    )
else:
    zone_occ = pd.DataFrame(columns=[
        "view_id", "zone_name", "frames_with_people",
        "mean_people_when_occupied", "peak_people",
        "total_person_frame_observations"
    ])
zone_occ.to_csv(out_dir / "zone_occupancy_summary.csv", index=False)

# ---------- OUTPUT 4: PER-VIEW OCCUPANCY & MOVEMENT ----------
view_summary = []
for view_id, group in df.groupby("view_id"):
    frame_counts = group.groupby("frame_index")["local_person_key"].nunique()
    # Speed uses consecutive observations of the same local track.
    speeds = []
    for _, person in group.groupby("local_person_key"):
        person = person.sort_values("frame_index")
        x = person["foot_x"].to_numpy()
        y = person["foot_y"].to_numpy()
        t = person["timestamp_sec"].to_numpy()
        if len(person) > 1:
            dt = np.diff(t)
            dist = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
            valid = np.isfinite(dt) & (dt > 0)
            speeds.extend((dist[valid] / dt[valid]).tolist())

    view_summary.append({
        "view_id": int(view_id),
        "detections": int(len(group)),
        "unique_local_tracks": int(group["local_person_key"].nunique()),
        "frames_observed": int(group["frame_index"].nunique()),
        "mean_people_per_observed_frame": float(frame_counts.mean()),
        "peak_people_in_frame": int(frame_counts.max()),
        "frames_with_3plus_people": int((frame_counts >= 3).sum()),
        "median_track_speed_px_per_sec": (
            float(np.median(speeds)) if speeds else np.nan
        ),
    })

view_summary = pd.DataFrame(view_summary)
view_summary.to_csv(
    out_dir / "view_occupancy_movement_summary.csv", index=False
)

# ---------- OUTPUT 5: FOOT-TRAFFIC HEATMAP SUMMARY ----------
heatmap_rows = []
for view_id, group in df.groupby("view_id"):
    bx = (group["foot_x"] // HEATMAP_BIN_PX).astype(int)
    by = (group["foot_y"] // HEATMAP_BIN_PX).astype(int)
    bins = pd.DataFrame({"bin_x": bx, "bin_y": by})
    counts = bins.value_counts().rename("observations").reset_index()

    heatmap_rows.append({
        "view_id": int(view_id),
        "detections": int(len(group)),
        "occupied_heatmap_bins": int(len(counts)),
        "peak_bin_observations": (
            int(counts["observations"].max()) if not counts.empty else 0
        ),
        "bin_size_px": HEATMAP_BIN_PX,
    })

heatmap_summary = pd.DataFrame(heatmap_rows)
heatmap_summary.to_csv(out_dir / "heatmap_summary.csv", index=False)

# Save detailed per-detection zone membership for audit/reuse.
pd.DataFrame(zone_detection_rows).to_csv(
    out_dir / "zone_detection_membership.csv", index=False
)

# ---------- OUTPUT 6: PRELIMINARY FLAGS ----------
flags = []
for row in view_summary.itertuples(index=False):
    flags.append({
        "view_id": row.view_id,
        "detections": row.detections,
        "mean_people_per_observed_frame":
            row.mean_people_per_observed_frame,
        "peak_people_in_frame": row.peak_people_in_frame,
        "preliminary_note": (
            "Review occupancy and movement; 10 seconds is too short "
            "for a reliable bottleneck conclusion."
        ),
    })

pd.DataFrame(flags).to_csv(
    out_dir / "preliminary_zone_flags.csv", index=False
)

# ---------- VALIDATION / REPORT ----------
print("ANALYTICS REGENERATED")
print("Output folder:", out_dir)
print("Input detections:", len(df))
print("Views:", sorted(df["view_id"].unique().tolist()))
print("Zones:", len(zone_specs))
print("Zone visits:", len(visits))
print("Zone membership rows:", len(zone_detection_rows))
print("\nFiles created:")
for path in sorted(out_dir.glob("*.csv")):
    print(f"  {path.name}: {sum(1 for _ in open(path, encoding='utf-8')) - 1} rows")

print("\nDWELL SUMMARY")
display(dwell.round(2))

print("\nPER-VIEW SUMMARY")
display(view_summary.round(2))



In [ ]:
from pathlib import Path
import shutil

source = Path("/content/vendora_analytics_outputs")
destination = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)

if not source.exists():
    raise FileNotFoundError(f"Analytics folder not found: {source}")

if destination.exists():
    raise FileExistsError(
        f"Destination already exists: {destination}\n"
        "Nothing was overwritten. Tell me if you want to save a new version."
    )

shutil.copytree(source, destination)

print("Saved analytics to:", destination)
print("\nVerified files:")
for file in sorted(destination.glob("*.csv")):
    print(f"{file.name}: {file.stat().st_size:,} bytes")

print("\nTotal CSV files:", len(list(destination.glob("*.csv"))))

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)

visits = pd.read_csv(folder / "zone_visits.csv")

print("Total zone visits:", len(visits))
print("\nLongest 15 visits:")
display(
    visits.sort_values("duration_sec", ascending=False)[
        [
            "view_id", "zone_name", "local_person_key",
            "start_frame", "end_frame",
            "start_sec", "end_sec",
            "observed_frames", "duration_sec"
        ]
    ].head(15).round(2)
)

print("\nVisits lasting over 3 seconds:")
print((visits["duration_sec"] > 3).sum())

print("\nVisits where duration and observed detections differ:")
visits["span_frames"] = (
    visits["end_frame"] - visits["start_frame"] + 1
)
display(
    visits[
        visits["span_frames"] > visits["observed_frames"]
    ][
        [
            "view_id", "zone_name", "local_person_key",
            "span_frames", "observed_frames", "duration_sec"
        ]
    ].sort_values("span_frames", ascending=False).head(15)
)

In [ ]:
import cv2
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

video_path = Path("/content/combined_cctv.mp4")
zone_path = Path("/content/combined_cctv__2__zones.json")
tracks_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

with open(zone_path, "r", encoding="utf-8") as f:
    zone_data = json.load(f)

df = pd.read_csv(tracks_path)

# Find View 5's Zone G polygon
view5 = next(v for v in zone_data["views"] if v["view_id"] == 5)
zone_g = next(z for z in view5["zones"] if z["name"] == "zone_G")
polygon = zone_g["points"]

if isinstance(polygon[0], dict):
    polygon = [[p["x"], p["y"]] for p in polygon]
polygon = [(int(p[0]), int(p[1])) for p in polygon]

# Existing long visit: V5_3, frames 960–1199
person = df[
    (df["view_id"] == 5) &
    (df["corrected_track_id"].astype(str) == "3") &
    (df["frame_index"].between(960, 1199))
].copy()

print("Matching detections:", len(person))
print("Frames:", person["frame_index"].min(), "to", person["frame_index"].max())

sample_frames = [
    int(person["frame_index"].min()),
    int(person["frame_index"].median()),
    int(person["frame_index"].max())
]

cap = cv2.VideoCapture(str(video_path))
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, frame = cap.read()
    if not ok:
        ax.set_title(f"Could not read frame {frame_no}")
        ax.axis("off")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pts = __import__("numpy").array(polygon, dtype="int32")
    overlay = frame.copy()
    cv2.fillPoly(overlay, [pts], (0, 255, 0))
    frame = cv2.addWeighted(overlay, 0.22, frame, 0.78, 0)
    cv2.polylines(frame, [pts], True, (255, 255, 0), 3)

    hit = person[person["frame_index"] == frame_no]
    for r in hit.itertuples():
        x1, y1, x2, y2 = map(int, [r.x1, r.y1, r.x2, r.y2])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 50, 50), 2)
        cv2.circle(frame, (int(r.foot_x), int(r.foot_y)), 6, (255, 0, 255), -1)
        cv2.putText(frame, "V5_3", (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 50, 50), 2)

    ax.imshow(frame)
    ax.set_title(f"Frame {frame_no} | t={frame_no / 24:.2f}s")
    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)

visits = pd.read_csv(folder / "zone_visits.csv")

visits["span_frames"] = (
    visits["end_frame"] - visits["start_frame"] + 1
)
visits["missing_frames_in_span"] = (
    visits["span_frames"] - visits["observed_frames"]
)
visits["has_gap"] = visits["missing_frames_in_span"] > 0

long_visits = visits[visits["duration_sec"] > 3].copy()

print("Visits longer than 3 seconds:", len(long_visits))
print("Long visits with missing frames:", int(long_visits["has_gap"].sum()))
print(
    "Long visits with no missing frames:",
    int((~long_visits["has_gap"]).sum())
)

print("\nLong visits by zone:")
display(
    long_visits.groupby(["view_id", "zone_name"])
    .agg(
        long_visits=("duration_sec", "size"),
        mean_duration_sec=("duration_sec", "mean"),
        max_duration_sec=("duration_sec", "max"),
        visits_with_gaps=("has_gap", "sum")
    )
    .reset_index()
    .round(2)
)

print("\nLongest visits with missing frames:")
display(
    long_visits[long_visits["has_gap"]]
    .sort_values("duration_sec", ascending=False)
    [[
        "view_id", "zone_name", "local_person_key",
        "duration_sec", "span_frames",
        "observed_frames", "missing_frames_in_span"
    ]]
    .head(10)
    .round(2)
)

In [ ]:
import pandas as pd
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)

visits = pd.read_csv(folder / "zone_visits.csv")

visits["span_frames"] = (
    visits["end_frame"] - visits["start_frame"] + 1
)
visits["missing_frames_in_span"] = (
    visits["span_frames"] - visits["observed_frames"]
)
visits["has_gap"] = visits["missing_frames_in_span"] > 0

# Include every long visit with a gap.
gap_cases = visits[
    (visits["duration_sec"] > 3) & visits["has_gap"]
].copy()
gap_cases["review_reason"] = "Long visit with missing frames"

# Include the longest gap-free visit for selected zones.
selected_zones = {
    (1, "zone_A"),
    (1, "zone_B"),
    (3, "zone_E"),
    (4, "zone_F"),
    (5, "zone_G"),
}

gap_free = visits[
    (visits["duration_sec"] > 3) & (~visits["has_gap"])
].copy()

selected_cases = []
for view_id, zone_name in selected_zones:
    subset = gap_free[
        (gap_free["view_id"] == view_id) &
        (gap_free["zone_name"] == zone_name)
    ]
    if not subset.empty:
        selected_cases.append(
            subset.nlargest(1, "duration_sec")
        )

if selected_cases:
    selected_cases = pd.concat(selected_cases, ignore_index=True)
    selected_cases["review_reason"] = "Longest gap-free visit in zone"
else:
    selected_cases = pd.DataFrame(columns=visits.columns.tolist() + ["review_reason"])

review = pd.concat([gap_cases, selected_cases], ignore_index=True)
review = review.drop_duplicates(
    subset=["view_id", "zone_name", "local_person_key", "start_frame", "end_frame"]
)
review = review.sort_values(
    ["has_gap", "duration_sec"], ascending=[False, False]
)

review_path = folder / "dwell_time_visual_review_queue.csv"
review.to_csv(review_path, index=False)

print("Review cases:", len(review))
print("Saved to:", review_path)
display(
    review[
        [
            "view_id", "zone_name", "local_person_key",
            "start_frame", "end_frame", "duration_sec",
            "missing_frames_in_span", "review_reason"
        ]
    ].round(2)
)

In [ ]:

import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

video_path = Path("/content/combined_cctv.mp4")
zone_path = Path("/content/combined_cctv__2__zones.json")
tracks_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

with open(zone_path, "r", encoding="utf-8") as f:
    zone_data = json.load(f)

tracks = pd.read_csv(tracks_path)
tracks["corrected_track_id"] = tracks["corrected_track_id"].astype(str)

view_id = 4
zone_name = "zone_F"
track_key = "V4_8"
start_frame, end_frame = 745, 959

view = next(v for v in zone_data["views"] if int(v["view_id"]) == view_id)
zone = next(z for z in view["zones"] if z["name"] == zone_name)
points = zone["points"]

if isinstance(points[0], dict):
    points = [[p["x"], p["y"]] for p in points]
polygon = np.array(points, dtype=np.int32)

person = tracks[
    (tracks["view_id"] == view_id) &
    (tracks["corrected_track_id"] == "8") &
    (tracks["frame_index"].between(start_frame, end_frame))
].copy()

observed = sorted(person["frame_index"].astype(int).unique())
missing = sorted(set(range(start_frame, end_frame + 1)) - set(observed))

# Start, middle, end, plus frames near any missing-frame gaps.
sample_frames = {
    start_frame,
    (start_frame + end_frame) // 2,
    end_frame,
}
for f in missing:
    sample_frames.update([max(start_frame, f - 1), f, min(end_frame, f + 1)])

sample_frames = sorted(sample_frames)
print("Track:", track_key, "| Zone:", zone_name)
print("Observed detections:", len(observed))
print("Missing frames:", missing)
print("Sample frames:", sample_frames)

cap = cv2.VideoCapture(str(video_path))
cols = 3
rows = (len(sample_frames) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, frame = cap.read()

    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    overlay = frame.copy()
    cv2.fillPoly(overlay, [polygon], (0, 255, 0))
    frame = cv2.addWeighted(overlay, 0.20, frame, 0.80, 0)
    cv2.polylines(frame, [polygon], True, (255, 255, 0), 3)

    hit = person[person["frame_index"] == frame_no]
    if not hit.empty:
        r = hit.iloc[0]
        x1, y1, x2, y2 = map(int, [r["x1"], r["y1"], r["x2"], r["y2"]])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 50, 50), 2)
        cv2.circle(frame, (int(r["foot_x"]), int(r["foot_y"])), 6, (255, 0, 255), -1)
        status = "DETECTED"
    else:
        status = "NO TRACK DETECTION"

    ax.imshow(frame)
    ax.set_title(f"Frame {frame_no} | {frame_no/24:.2f}s | {status}")
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)

queue_path = folder / "dwell_time_visual_review_queue.csv"
review = pd.read_csv(queue_path)

review["visual_review_status"] = "not_reviewed"
review["visual_review_notes"] = ""

mask = (
    (review["view_id"] == 4) &
    (review["zone_name"] == "zone_F") &
    (review["local_person_key"] == "V4_8")
)

review.loc[mask, "visual_review_status"] = "tracking_unreliable"
review.loc[mask, "visual_review_notes"] = (
    "Visual inspection: tracker loses/switches the person; "
    "do not treat full visit span as validated shopper dwell."
)

review_path = folder / "dwell_time_visual_review_queue_v2.csv"
review.to_csv(review_path, index=False)

print("Saved:", review_path)
print("\nRecorded case:")
display(
    review.loc[
        mask,
        [
            "view_id", "zone_name", "local_person_key",
            "duration_sec", "visual_review_status",
            "visual_review_notes"
        ]
    ]
)

In [ ]:

import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

video_path = Path("/content/combined_cctv.mp4")
zone_path = Path("/content/combined_cctv__2__zones.json")
tracks_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

with open(zone_path, "r", encoding="utf-8") as f:
    zone_data = json.load(f)

tracks = pd.read_csv(tracks_path)
tracks["corrected_track_id"] = tracks["corrected_track_id"].astype(str)

view_id = 3
zone_name = "zone_E"
start_frame, end_frame = 523, 719

view = next(v for v in zone_data["views"] if int(v["view_id"]) == view_id)
zone = next(z for z in view["zones"] if z["name"] == zone_name)
points = zone["points"]

if isinstance(points[0], dict):
    points = [[p["x"], p["y"]] for p in points]
polygon = np.array(points, dtype=np.int32)

person = tracks[
    (tracks["view_id"] == view_id) &
    (tracks["corrected_track_id"] == "11") &
    (tracks["frame_index"].between(start_frame, end_frame))
].copy()

observed = sorted(person["frame_index"].astype(int).unique())
missing = sorted(set(range(start_frame, end_frame + 1)) - set(observed))

sample_frames = {
    start_frame,
    (start_frame + end_frame) // 2,
    end_frame,
}
for f in missing:
    sample_frames.update([
        max(start_frame, f - 1),
        f,
        min(end_frame, f + 1)
    ])

sample_frames = sorted(sample_frames)
print("Track: V3_11 | Zone: zone_E")
print("Observed detections:", len(observed))
print("Missing frames:", missing)
print("Sample frames:", sample_frames)

cap = cv2.VideoCapture(str(video_path))
cols = 3
rows = (len(sample_frames) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, frame = cap.read()

    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    overlay = frame.copy()
    cv2.fillPoly(overlay, [polygon], (0, 255, 0))
    frame = cv2.addWeighted(overlay, 0.20, frame, 0.80, 0)
    cv2.polylines(frame, [polygon], True, (255, 255, 0), 3)

    hit = person[person["frame_index"] == frame_no]
    if not hit.empty:
        r = hit.iloc[0]
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 50, 50), 2)
        cv2.circle(
            frame, (int(r["foot_x"]), int(r["foot_y"])),
            6, (255, 0, 255), -1
        )
        status = "DETECTED"
    else:
        status = "NO TRACK DETECTION"

    ax.imshow(frame)
    ax.set_title(f"Frame {frame_no} | {frame_no/24:.2f}s | {status}")
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

# Locate V3_11 in Zone E without changing any other review records.
text_cols = df.select_dtypes(include="object").columns
mask = (
    df[text_cols].fillna("").astype(str).eq("V3_11").any(axis=1)
    & df[text_cols].fillna("").astype(str).eq("zone_E").any(axis=1)
)

if mask.sum() != 1:
    print("Could not identify exactly one matching row.")
    print("Columns:", df.columns.tolist())
    print(df[df[text_cols].fillna("").astype(str)
             .apply(lambda col: col.str.contains("V3_11", regex=False))
             .any(axis=1)].to_string(index=False))
else:
    df.loc[mask, "visual_review_status"] = "tracking_unreliable"
    df.loc[mask, "visual_review_notes"] = (
        "Visual inspection: tracker loses/switches the person; "
        "do not treat the full visit span as validated shopper dwell."
    )

    df.to_csv(path, index=False)
    print("Updated review record successfully.")
    print(df.loc[mask].to_string(index=False))


In [ ]:

import cv2
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

VIDEO_PATH = "/content/combined_cctv.mp4"
ZONE_JSON = "/content/combined_cctv__2__zones.json"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

TARGET_VIEW = 6
TARGET_TRACK = "29"
ZONE_NAME = "zone_J"
START_FRAME = 1272
END_FRAME = 1421

# Load existing annotations and corrected tracks; do not rerun detection/tracking.
tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

print("Available zone JSON keys:", list(zone_data.keys()))

# Find the target zone polygon, accommodating common JSON structures.
def find_zone(obj, target_name):
    if isinstance(obj, dict):
        if obj.get("name") == target_name or obj.get("zone_name") == target_name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, target_name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, target_name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)

if raw_polygon is None:
    raise ValueError(
        f"Could not automatically locate {ZONE_NAME}. "
        "Inspect the printed JSON keys/structure before proceeding."
    )

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

# Select the existing corrected track and its visit span.
view_col = "view_id"
id_col = "corrected_track_id"
frame_col = "frame_index"

case = tracks[
    (tracks[view_col] == TARGET_VIEW) &
    (tracks[id_col].astype(str) == TARGET_TRACK) &
    (tracks[frame_col].between(START_FRAME, END_FRAME))
].copy()

if case.empty:
    raise ValueError(
        "No rows found for V6_29. Check the corrected ID and CSV columns."
    )

observed = set(case[frame_col].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)

print(f"V6_29 / {ZONE_NAME}: frames {START_FRAME}–{END_FRAME}")
print(f"Observed rows: {len(case)}")
print(f"Missing frames in span: {len(missing)}")
print("Missing frame indices:", missing)

# Sample the start, midpoint, end, plus frames near each gap.
sample_frames = {
    START_FRAME,
    (START_FRAME + END_FRAME) // 2,
    END_FRAME
}
for m in missing:
    sample_frames.update(
        f for f in range(max(START_FRAME, m - 2),
                         min(END_FRAME, m + 2) + 1)
    )

sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

# Expected view 6 occupies seconds 50–60, i.e. frames 1200–1439.
# Show all visible person boxes in the target view, not just ID 29,
# so identity switches and nearby people can be assessed.
view_tracks = tracks[
    (tracks[view_col] == TARGET_VIEW) &
    (tracks[frame_col].between(START_FRAME, END_FRAME))
].copy()

fig, axes = plt.subplots(
    nrows=int(np.ceil(len(sample_frames) / 2)),
    ncols=2, figsize=(16, 5 * int(np.ceil(len(sample_frames) / 2)))
)
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_no))
    ok, frame = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: read failed")
        ax.axis("off")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    cv2.polylines(frame, [polygon], True, (255, 255, 0), 3)

    fr = view_tracks[view_tracks[frame_col] == frame_no]
    target_present = False

    for _, r in fr.iterrows():
        x1, y1, x2, y2 = [int(r[c]) for c in ["x1", "y1", "x2", "y2"]]
        tid = str(r[id_col])
        is_target = tid == TARGET_TRACK
        color = (255, 40, 40) if is_target else (40, 220, 80)

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            frame, f"ID {tid}",
            (x1, max(20, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2
        )
        if is_target:
            target_present = True

    ax.imshow(frame)
    ax.set_title(
        f"Frame {frame_no} | target ID 29: "
        f"{'VISIBLE' if target_present else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

mask = (
    (df["view_id"] == 6)
    & (df["local_person_key"].astype(str) == "V6_29")
    & (df["zone_name"] == "zone_J")
)

if mask.sum() != 1:
    raise ValueError(f"Expected one V6_29 / zone_J row; found {mask.sum()}")

df.loc[mask, "visual_review_status"] = "localized_id_switch"
df.loc[mask, "visual_review_notes"] = (
    "Visual review: sampled frames appear correct except frame 1325, "
    "where the tracked ID changes from 29 to 46. Localized ID switch; "
    "do not merge IDs automatically. Verify identity continuity around "
    "frame 1325 before treating the entire dwell span as confirmed."
)

df.to_csv(path, index=False)

print("Updated V6_29 review record:")
print(df.loc[mask].to_string(index=False))


In [ ]:

import cv2
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

VIDEO_PATH = "/content/combined_cctv.mp4"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

df = pd.read_csv(TRACKS_CSV)
cap = cv2.VideoCapture(VIDEO_PATH)

frames_to_show = list(range(1321, 1331))
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for ax, frame_no in zip(axes, frames_to_show):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()

    if not ok:
        ax.set_title(f"Frame {frame_no}: read failed")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rows = df[
        (df["view_id"] == 6)
        & (df["frame_index"] == frame_no)
        & (df["corrected_track_id"].astype(str).isin(["29", "46"]))
    ]

    for _, r in rows.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        color = (255, 50, 50) if tid == "29" else (50, 220, 80)

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 3)
        cv2.putText(
            image, f"ID {tid}", (x1, max(25, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2
        )

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | "
        f"ID 29 present: {any(rows['corrected_track_id'].astype(str) == '29')} | "
        f"ID 46 present: {any(rows['corrected_track_id'].astype(str) == '46')}"
    )
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

mask = (
    (df["view_id"] == 6)
    & (df["local_person_key"].astype(str) == "V6_29")
    & (df["zone_name"] == "zone_J")
)

if mask.sum() != 1:
    raise ValueError(f"Expected one matching row; found {mask.sum()}")

df.loc[mask, "visual_review_status"] = "confirmed_local_id_switch"
df.loc[mask, "visual_review_notes"] = (
    "Visual review confirms the same physical shopper continues while "
    "the tracking ID switches from 29 to 46 around frame 1325. "
    "Treat as a localized ID switch for review; original tracking data "
    "remains unchanged. Any corrected dwell calculation must account "
    "for the identity continuity and zone membership."
)

df.to_csv(path, index=False)

print(df.loc[mask].to_string(index=False))
print("\nReview queue updated. Original tracking CSV remains unchanged.")


In [ ]:

import cv2
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

VIDEO_PATH = "/content/combined_cctv.mp4"
ZONE_JSON = "/content/combined_cctv__2__zones.json"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

# Load existing files
tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

# Locate Zone J polygon recursively
def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            points = obj.get(
                "points", obj.get("polygon", obj.get("coordinates"))
            )
            if points is not None:
                return points
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, "zone_J")
if raw_polygon is None:
    raise ValueError(
        "Zone J polygon wasn't found. Inspect the JSON structure before proceeding."
    )

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

# Check frames surrounding the confirmed ID switch
frames = list(range(1321, 1331))
view_tracks = tracks[
    (tracks["view_id"] == 6)
    & (tracks["frame_index"].isin(frames))
    & (tracks["corrected_track_id"].astype(str).isin(["29", "46"]))
].copy()

# Foot-point zone membership table
results = []
for _, r in view_tracks.iterrows():
    x = float(r["foot_x"])
    y = float(r["foot_y"])
    inside = cv2.pointPolygonTest(
        polygon.astype(np.float32), (x, y), False
    ) >= 0

    results.append({
        "frame_index": int(r["frame_index"]),
        "track_id": str(r["corrected_track_id"]),
        "foot_x": x,
        "foot_y": y,
        "inside_zone_J": inside
    })

membership = pd.DataFrame(results)
print("Zone J membership around the ID switch:")
if membership.empty:
    print("No rows found for IDs 29/46 in frames 1321–1330.")
else:
    display(membership.sort_values(["frame_index", "track_id"]))

# Show video frames with polygon and tracked boxes
cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for ax, frame_no in zip(axes, frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()

    if not ok:
        ax.set_title(f"Frame {frame_no}: unable to read")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    rows = view_tracks[view_tracks["frame_index"] == frame_no]
    labels = []

    for _, r in rows.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        foot = (int(r["foot_x"]), int(r["foot_y"]))
        inside = cv2.pointPolygonTest(
            polygon.astype(np.float32), foot, False
        ) >= 0

        color = (255, 50, 50) if tid == "29" else (50, 220, 80)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 3)
        cv2.circle(image, foot, 6, color, -1)
        cv2.putText(
            image, f"ID {tid} | {'IN' if inside else 'OUT'}",
            (x1, max(25, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2
        )
        labels.append(f"ID {tid}: {'IN' if inside else 'OUT'}")

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no}: " + (", ".join(labels) if labels else "IDs 29/46 not detected")
    )
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

print("Review status counts:")
display(df["visual_review_status"].fillna("pending").value_counts().to_frame("cases"))

print("\nCases not yet reviewed or requiring follow-up:")
reviewed_statuses = {
    "tracking_unreliable",
    "confirmed_local_id_switch",
    "localized_id_switch",
}

pending = df[
    ~df["visual_review_status"].fillna("").isin(reviewed_statuses)
]

display(
    pending[
        [
            "view_id", "zone_name", "local_person_key",
            "start_frame", "end_frame", "duration_sec",
            "missing_frames_in_span", "visual_review_status"
        ]
    ].sort_values("duration_sec", ascending=False)
)


In [ ]:

import pandas as pd

path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

mask = (
    (df["view_id"] == 5)
    & (df["local_person_key"].astype(str) == "V5_3")
    & (df["zone_name"] == "zone_G")
)

if mask.sum() != 1:
    raise ValueError(f"Expected one V5_3 / zone_G row; found {mask.sum()}")

df.loc[mask, "visual_review_status"] = "plausible_browsing_unconfirmed"
df.loc[mask, "visual_review_notes"] = (
    "Visual review: person stays near the shelf and appears to browse. "
    "Continuous identity throughout the full 10-second span is not "
    "independently confirmed; treat dwell as plausible, not verified."
)

df.to_csv(path, index=False)

print("Updated V5_3 review record:")
print(df.loc[mask].to_string(index=False))


In [ ]:

import cv2
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/combined_cctv.mp4"
ZONE_JSON = "/content/combined_cctv__2__zones.json"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

VIEW_ID = 1
TRACK_ID = "6"
ZONE_NAME = "zone_A"
START_FRAME = 12
END_FRAME = 224

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(
        f"Could not find {ZONE_NAME} polygon. Inspect the JSON structure."
    )

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & (tracks["frame_index"].between(START_FRAME, END_FRAME))
].copy()

print(f"V1_{TRACK_ID} / {ZONE_NAME}")
print(f"Rows for this ID in the span: {len(case)}")
print(f"Expected frame span: {END_FRAME - START_FRAME + 1}")

# Sample frames across the visit, including early, middle and late stages.
sample_frames = sorted(set([
    START_FRAME, 30, 50, 75, 100, 125, 150, 175, 200, END_FRAME
]))

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()

    if not ok:
        ax.set_title(f"Frame {frame_no}: unable to read")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    fr = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_present = False
    for _, r in fr.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        is_target = tid == TRACK_ID
        color = (255, 50, 50) if is_target else (50, 220, 80)

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        if is_target:
            target_present = True

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 6: "
        f"{'VISIBLE' if target_present else 'NOT DETECTED'}"
    )
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

mask = (
    (df["view_id"] == 1)
    & (df["local_person_key"].astype(str) == "V1_6")
    & (df["zone_name"] == "zone_A")
)

if mask.sum() != 1:
    raise ValueError(f"Expected one V1_6 / zone_A row; found {mask.sum()}")

df.loc[mask, "visual_review_status"] = "visually_supported_gap_free"
df.loc[mask, "visual_review_notes"] = (
    "Visual review: ID 6 remains on the same physical person across "
    "sampled frames. Person remains inside Zone A after entering the "
    "shop. No missing detection frames in the recorded visit span; "
    "continuous identity is supported by sampled-frame inspection."
)

df.to_csv(path, index=False)

print("Saved review result:")
print(df.loc[mask].to_string(index=False))


In [ ]:

import cv2
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/combined_cctv.mp4"
ZONE_JSON = "/content/combined_cctv__2__zones.json"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

VIEW_ID, TRACK_ID, ZONE_NAME = 4, "10", "zone_F"
START_FRAME, END_FRAME = 748, 959

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find {ZONE_NAME} polygon in the JSON.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & (tracks["frame_index"].between(START_FRAME, END_FRAME))
].copy()

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)
print(f"V4_{TRACK_ID} / {ZONE_NAME}")
print(f"Observed rows: {len(case)}")
print(f"Missing frames in span: {len(missing)}")

sample_frames = sorted(set([
    START_FRAME, 770, 790, 810, 830,
    850, 875, 900, 930, END_FRAME
]))

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unable to read")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    fr = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_present = False
    for _, r in fr.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        is_target = tid == TRACK_ID
        color = (255, 50, 50) if is_target else (50, 220, 80)

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        if is_target:
            target_present = True

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 10: "
        f"{'VISIBLE' if target_present else 'NOT DETECTED'}"
    )
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

mask = (
    (df["view_id"] == 4)
    & (df["local_person_key"].astype(str) == "V4_10")
    & (df["zone_name"] == "zone_F")
)

if mask.sum() != 1:
    raise ValueError(f"Expected one V4_10 / zone_F row; found {mask.sum()}")

df.loc[mask, "visual_review_status"] = "visually_supported_gap_free"
df.loc[mask, "visual_review_notes"] = (
    "Visual review: ID 10 remains on the same physical person across "
    "sampled frames, and the person remains inside Zone F. No missing "
    "detection frames in the recorded visit span; continuous identity "
    "is supported by sampled-frame inspection."
)

df.to_csv(path, index=False)

print(df.loc[mask].to_string(index=False))


In [ ]:

import cv2
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/combined_cctv.mp4"
ZONE_JSON = "/content/combined_cctv__2__zones.json"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

VIEW_ID, TRACK_ID, ZONE_NAME = 1, "5", "zone_B"
START_FRAME, END_FRAME = 0, 211

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find {ZONE_NAME} polygon.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & (tracks["frame_index"].between(START_FRAME, END_FRAME))
].copy()

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)
print(f"V1_{TRACK_ID} / {ZONE_NAME}")
print(f"Observed rows: {len(case)}")
print(f"Missing frames in span: {len(missing)}")

sample_frames = [0, 20, 45, 70, 95, 120, 145, 170, 190, 211]

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unable to read")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    fr = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_present = False
    for _, r in fr.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        is_target = tid == TRACK_ID
        color = (255, 50, 50) if is_target else (50, 220, 80)

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        if is_target:
            target_present = True

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 5: "
        f"{'VISIBLE' if target_present else 'NOT DETECTED'}"
    )
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

mask = (
    (df["view_id"] == 1)
    & (df["local_person_key"].astype(str) == "V1_5")
    & (df["zone_name"] == "zone_B")
)

if mask.sum() != 1:
    raise ValueError(f"Expected one V1_5 / zone_B row; found {mask.sum()}")

df.loc[mask, "visual_review_status"] = "visually_supported_gap_free"
df.loc[mask, "visual_review_notes"] = (
    "Visual review: ID 5 remains on the same physical person across "
    "sampled frames, and the person remains inside Zone B. No missing "
    "detection frames in the recorded visit span; continuous identity "
    "is supported by sampled-frame inspection."
)

df.to_csv(path, index=False)

print(df.loc[mask].to_string(index=False))


In [ ]:

import cv2
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/combined_cctv.mp4"
ZONE_JSON = "/content/combined_cctv__2__zones.json"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

VIEW_ID, TRACK_ID, ZONE_NAME = 3, "1", "zone_E"
START_FRAME, END_FRAME = 480, 670

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find {ZONE_NAME} polygon.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & (tracks["frame_index"].between(START_FRAME, END_FRAME))
].copy()

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)

print(f"V3_{TRACK_ID} / {ZONE_NAME}")
print(f"Observed rows: {len(case)}")
print(f"Missing frames in span: {len(missing)}")

sample_frames = [480, 500, 520, 540, 560, 580, 600, 620, 645, 670]

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()

    if not ok:
        ax.set_title(f"Frame {frame_no}: unable to read")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    fr = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_present = False
    for _, r in fr.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        is_target = tid == TRACK_ID
        color = (255, 50, 50) if is_target else (50, 220, 80)

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        if is_target:
            target_present = True

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 1: "
        f"{'VISIBLE' if target_present else 'NOT DETECTED'}"
    )
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/dwell_time_visual_review_queue_v2.csv"
)

df = pd.read_csv(path)

mask = (
    (df["view_id"] == 3)
    & (df["local_person_key"].astype(str) == "V3_1")
    & (df["zone_name"] == "zone_E")
)

if mask.sum() != 1:
    raise ValueError(f"Expected one V3_1 / zone_E row; found {mask.sum()}")

df.loc[mask, "visual_review_status"] = "visually_supported_gap_free"
df.loc[mask, "visual_review_notes"] = (
    "Visual review: ID 1 remains on the same physical person across "
    "sampled frames, and the person remains inside Zone E. No missing "
    "detection frames in the recorded visit span; continuous identity "
    "is supported by sampled-frame inspection."
)

df.to_csv(path, index=False)

print(df.loc[mask].to_string(index=False))


In [ ]:

import cv2
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/combined_cctv.mp4"
ZONE_JSON = "/content/combined_cctv__2__zones.json"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

VIEW_ID, TRACK_ID, ZONE_NAME = 5, "7", "zone_G"
START_FRAME, END_FRAME = 988, 1136

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find {ZONE_NAME} polygon.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & (tracks["frame_index"].between(START_FRAME, END_FRAME))
].copy()

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)

print(f"V5_{TRACK_ID} / {ZONE_NAME}")
print(f"Observed rows: {len(case)}")
print(f"Missing frames in span: {missing}")

# Inspect the start, middle, end and frames surrounding any missing detection.
sample_frames = {
    START_FRAME, 1010, 1035, 1060, 1085, 1110, END_FRAME
}
for m in missing:
    sample_frames.update(
        f for f in range(max(START_FRAME, m - 3),
                         min(END_FRAME, m + 3) + 1)
    )
sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
fig, axes = plt.subplots(
    int(np.ceil(len(sample_frames) / 2)), 2,
    figsize=(16, 5 * int(np.ceil(len(sample_frames) / 2)))
)
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unable to read")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    fr = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_present = False
    for _, r in fr.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        is_target = tid == TRACK_ID
        color = (255, 50, 50) if is_target else (50, 220, 80)

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        if is_target:
            target_present = True

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 7: "
        f"{'VISIBLE' if target_present else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

print("Columns in review CSV:")
print(review.columns.tolist())

print("\nFirst 3 rows:")
display(review.head(3))


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

mask = (
    (review["view_id"] == 5)
    & (review["zone_name"] == "zone_G")
    & (review["local_person_key"] == "V5_7")
)

if mask.sum() != 1:
    print(f"Expected exactly 1 matching row; found {mask.sum()}.")
    print(review.loc[
        (review["view_id"] == 5) & (review["zone_name"] == "zone_G"),
        ["local_person_key", "start_frame", "end_frame"]
    ])
else:
    review.loc[mask, "visual_review_status"] = "visually_supported_with_gap"
    review.loc[mask, "visual_review_notes"] = (
        "Visual review supports the same physical person remaining inside "
        "Zone G across sampled frames, including around the missing detection. "
        "One detection frame is missing, so continuous occupancy during the "
        "gap is not confirmed. Original tracking data remains unchanged."
    )

    review.to_csv(review_path, index=False)

    print("Updated review entry:")
    display(review.loc[mask, [
        "view_id", "zone_name", "local_person_key",
        "start_frame", "end_frame", "duration_sec",
        "missing_frames_in_span",
        "visual_review_status", "visual_review_notes"
    ]])
    print("\nSaved successfully.")


In [ ]:

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/combined_cctv.mp4"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)

VIEW_ID = 2
LOCAL_KEY = "V2_5"
TRACK_ID = "5"
ZONE_NAME = "zone_C"
START_FRAME, END_FRAME = 328, 457

tracks = pd.read_csv(TRACKS_CSV)

# Load the existing polygon JSON.
import json
with open("/content/combined_cctv__2__zones.json", "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find polygon for {ZONE_NAME}.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & tracks["frame_index"].between(START_FRAME, END_FRAME)
].copy()

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)

print(f"{LOCAL_KEY} | {ZONE_NAME} | frames {START_FRAME}-{END_FRAME}")
print("Missing frames:", missing)

sample_frames = {
    START_FRAME, 345, 365, 385, 405, 425, 445, END_FRAME
}
for m in missing:
    sample_frames.update(
        range(max(START_FRAME, m - 3), min(END_FRAME, m + 3) + 1)
    )
sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
rows = int(np.ceil(len(sample_frames) / 2))
fig, axes = plt.subplots(rows, 2, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    frame_tracks = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_visible = False
    for _, r in frame_tracks.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        target = tid == TRACK_ID
        color = (255, 50, 50) if target else (50, 220, 80)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        target_visible |= target

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 5: "
        f"{'VISIBLE' if target_visible else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.releas
plt.show()


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

mask = (
    (review["view_id"] == 2)
    & (review["zone_name"] == "zone_C")
    & (review["local_person_key"] == "V2_5")
)

if mask.sum() != 1:
    print(f"Expected 1 matching row, found {mask.sum()}.")
else:
    review.loc[mask, "visual_review_status"] = "tracking_unreliable"
    review.loc[mask, "visual_review_notes"] = (
        "Visual review: occlusion occurs around frame 364, after which "
        "the tracker follows a different physical person. Do not treat "
        "the full visit span as one shopper's dwell time. Original "
        "tracking data remains unchanged."
    )

    review.to_csv(review_path, index=False)
    display(review.loc[mask, [
        "view_id", "zone_name", "local_person_key",
        "start_frame", "end_frame", "duration_sec",
        "visual_review_status", "visual_review_notes"
    ]])
    print("Saved successfully.")


In [ ]:

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

VIDEO_PATH = "/content/combined_cctv.mp4"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)
ZONE_JSON = "/content/combined_cctv__2__zones.json"

VIEW_ID, TRACK_ID, ZONE_NAME = 1, "2", "zone_B"
START_FRAME, END_FRAME = 0, 110

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find polygon for {ZONE_NAME}.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & tracks["frame_index"].between(START_FRAME, END_FRAME)
]

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)
print("V1_2 | Zone B")
print("Missing frames:", missing)

sample_frames = {0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110}
for m in missing:
    sample_frames.update(
        range(max(START_FRAME, m - 3), min(END_FRAME, m + 3) + 1)
    )
sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
rows = int(np.ceil(len(sample_frames) / 2))
fig, axes = plt.subplots(rows, 2, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    frame_tracks = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_visible = False
    for _, r in frame_tracks.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        target = tid == TRACK_ID
        color = (255, 50, 50) if target else (50, 220, 80)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        target_visible |= target

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 2: "
        f"{'VISIBLE' if target_visible else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

mask = (
    (review["view_id"] == 1)
    & (review["zone_name"] == "zone_B")
    & (review["local_person_key"] == "V1_2")
)

if mask.sum() != 1:
    print(f"Expected 1 matching row, found {mask.sum()}.")
else:
    review.loc[mask, "visual_review_status"] = "visually_supported_with_gap"
    review.loc[mask, "visual_review_notes"] = (
        "Visual review supports the same physical shopper remaining inside "
        "Zone B across sampled frames. Two detection frames are missing, "
        "so continuous occupancy during those gaps is not confirmed. "
        "Original tracking data remains unchanged."
    )

    review.to_csv(review_path, index=False)
    display(review.loc[mask, [
        "view_id", "zone_name", "local_person_key",
        "start_frame", "end_frame", "duration_sec",
        "missing_frames_in_span", "visual_review_status"
    ]])
    print("Saved successfully.")


In [ ]:

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

VIDEO_PATH = "/content/combined_cctv.mp4"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)
ZONE_JSON = "/content/combined_cctv__2__zones.json"

VIEW_ID, TRACK_ID, ZONE_NAME = 3, "10", "zone_E"
START_FRAME, END_FRAME = 562, 658

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find polygon for {ZONE_NAME}.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & tracks["frame_index"].between(START_FRAME, END_FRAME)
]

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)
print("V3_10 | Zone E")
print("Missing frames:", missing)

sample_frames = {562, 575, 588, 601, 614, 627, 640, 658}
for m in missing:
    sample_frames.update(
        range(max(START_FRAME, m - 3), min(END_FRAME, m + 3) + 1)
    )
sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
rows = int(np.ceil(len(sample_frames) / 2))
fig, axes = plt.subplots(rows, 2, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    frame_tracks = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_visible = False
    for _, r in frame_tracks.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        target = tid == TRACK_ID
        color = (255, 50, 50) if target else (50, 220, 80)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        target_visible |= target

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 10: "
        f"{'VISIBLE' if target_visible else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

mask = (
    (review["view_id"] == 3)
    & (review["zone_name"] == "zone_E")
    & (review["local_person_key"] == "V3_10")
)

if mask.sum() != 1:
    print(f"Expected 1 matching row, found {mask.sum()}.")
else:
    review.loc[mask, "visual_review_status"] = "tracking_unreliable"
    review.loc[mask, "visual_review_notes"] = (
        "Visual review: the tracked ID changes from 10 to 33 around "
        "frames 640-647. Identity continuity for the full visit is "
        "unreliable; do not count the entire span as validated dwell "
        "for one shopper. Original tracking data remains unchanged."
    )

    review.to_csv(review_path, index=False)
    display(review.loc[mask, [
        "view_id", "zone_name", "local_person_key",
        "start_frame", "end_frame", "duration_sec",
        "missing_frames_in_span", "visual_review_status"
    ]])
    print("Saved successfully.")


In [ ]:

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

VIDEO_PATH = "/content/combined_cctv.mp4"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)
ZONE_JSON = "/content/combined_cctv__2__zones.json"

VIEW_ID, TRACK_ID, ZONE_NAME = 6, "49", "zone_I"
START_FRAME, END_FRAME = 1319, 1406

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find polygon for {ZONE_NAME}.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & tracks["frame_index"].between(START_FRAME, END_FRAME)
]

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)
print("V6_49 | Zone I")
print("Missing frames:", missing)

sample_frames = {1319, 1330, 1342, 1354, 1366, 1378, 1390, 1406}
for m in missing:
    sample_frames.update(
        range(max(START_FRAME, m - 3), min(END_FRAME, m + 3) + 1)
    )
sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
rows = int(np.ceil(len(sample_frames) / 2))
fig, axes = plt.subplots(rows, 2, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    frame_tracks = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_visible = False
    for _, r in frame_tracks.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        target = tid == TRACK_ID
        color = (255, 50, 50) if target else (50, 220, 80)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        target_visible |= target

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 49: "
        f"{'VISIBLE' if target_visible else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

mask = (
    (review["view_id"] == 6)
    & (review["zone_name"] == "zone_I")
    & (review["local_person_key"] == "V6_49")
)

if mask.sum() != 1:
    print(f"Expected 1 matching row, found {mask.sum()}.")
else:
    review.loc[mask, "visual_review_status"] = "visually_supported_with_gap"
    review.loc[mask, "visual_review_notes"] = (
        "Visual review supports the same physical shopper remaining inside "
        "Zone I across sampled frames. One detection frame is missing, so "
        "continuous occupancy during the gap is not confirmed. Original "
        "tracking data remains unchanged."
    )

    review.to_csv(review_path, index=False)
    display(review.loc[mask, [
        "view_id", "zone_name", "local_person_key",
        "start_frame", "end_frame", "duration_sec",
        "missing_frames_in_span", "visual_review_status"
    ]])
    print("Saved successfully.")


In [ ]:

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

VIDEO_PATH = "/content/combined_cctv.mp4"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)
ZONE_JSON = "/content/combined_cctv__2__zones.json"

VIEW_ID, TRACK_ID, ZONE_NAME = 4, "1", "zone_F"
START_FRAME, END_FRAME = 720, 801

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find polygon for {ZONE_NAME}.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & tracks["frame_index"].between(START_FRAME, END_FRAME)
]

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)
print("V4_1 | Zone F")
print("Missing frames:", missing)

sample_frames = {720, 730, 740, 750, 760, 770, 780, 790, 801}
for m in missing:
    sample_frames.update(
        range(max(START_FRAME, m - 3), min(END_FRAME, m + 3) + 1)
    )
sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
rows = int(np.ceil(len(sample_frames) / 2))
fig, axes = plt.subplots(rows, 2, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    frame_tracks = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_visible = False
    for _, r in frame_tracks.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        target = tid == TRACK_ID
        color = (255, 50, 50) if target else (50, 220, 80)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        target_visible |= target

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 1: "
        f"{'VISIBLE' if target_visible else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

mask = (
    (review["view_id"] == 4)
    & (review["zone_name"] == "zone_F")
    & (review["local_person_key"] == "V4_1")
)

if mask.sum() != 1:
    print(f"Expected 1 matching row, found {mask.sum()}.")
else:
    review.loc[mask, "visual_review_status"] = "confirmed_local_id_switch"
    review.loc[mask, "visual_review_notes"] = (
        "Visual review confirms the same physical shopper switches between "
        "tracking IDs 1 and 3 within Zone F. The original ID 1 visit should "
        "not be interpreted as a fully reliable standalone identity span. "
        "Original tracking data remains unchanged."
    )

    review.to_csv(review_path, index=False)
    display(review.loc[mask, [
        "view_id", "zone_name", "local_person_key",
        "start_frame", "end_frame", "duration_sec",
        "missing_frames_in_span", "visual_review_status"
    ]])
    print("Saved successfully.")


In [ ]:

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

VIDEO_PATH = "/content/combined_cctv.mp4"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)
ZONE_JSON = "/content/combined_cctv__2__zones.json"

VIEW_ID, TRACK_ID, ZONE_NAME = 6, "2", "zone_J"
START_FRAME, END_FRAME = 1200, 1274

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find polygon for {ZONE_NAME}.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & tracks["frame_index"].between(START_FRAME, END_FRAME)
]

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)
print("V6_2 | Zone J")
print("Missing frames:", missing)

sample_frames = {1200, 1210, 1220, 1230, 1240, 1250, 1260, 1274}
for m in missing:
    sample_frames.update(
        range(max(START_FRAME, m - 3), min(END_FRAME, m + 3) + 1)
    )
sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
rows = int(np.ceil(len(sample_frames) / 2))
fig, axes = plt.subplots(rows, 2, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    frame_tracks = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_visible = False
    for _, r in frame_tracks.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        target = tid == TRACK_ID
        color = (255, 50, 50) if target else (50, 220, 80)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        target_visible |= target

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 2: "
        f"{'VISIBLE' if target_visible else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

mask = (
    (review["view_id"] == 6)
    & (review["zone_name"] == "zone_J")
    & (review["local_person_key"] == "V6_2")
)

if mask.sum() != 1:
    print(f"Expected 1 matching row, found {mask.sum()}.")
else:
    review.loc[mask, "visual_review_status"] = "tracking_unreliable"
    review.loc[mask, "visual_review_notes"] = (
        "Visual review: after an occlusion around frame 1265, the tracker "
        "switches to a different physical person. Do not treat the full "
        "visit span as one shopper's validated dwell. Original tracking "
        "data remains unchanged."
    )

    review.to_csv(review_path, index=False)
    display(review.loc[mask, [
        "view_id", "zone_name", "local_person_key",
        "start_frame", "end_frame", "duration_sec",
        "missing_frames_in_span", "visual_review_status"
    ]])
    print("Saved successfully.")


In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

# Find the case that still needs review.
pending = review[
    review["visual_review_status"].fillna("").str.strip().isin(
        ["", "not_reviewed"]
    )
]

print("Cases still needing review:", len(pending))
display(
    pending[
        [
            "view_id", "zone_name", "local_person_key",
            "start_frame", "end_frame", "duration_sec",
            "missing_frames_in_span", "visual_review_status",
            "visual_review_notes"
        ]
    ]
)

print("\nFull review queue:")
display(
    review.sort_values(["view_id", "start_frame"])[
        [
            "view_id", "zone_name", "local_person_key",
            "start_frame", "end_frame", "duration_sec",
            "missing_frames_in_span", "visual_review_status"
        ]
    ]
)


In [ ]:

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

VIDEO_PATH = "/content/combined_cctv.mp4"
TRACKS_CSV = (
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "all_views_tracks_manual_corrections_v5.csv"
)
ZONE_JSON = "/content/combined_cctv__2__zones.json"

VIEW_ID, TRACK_ID, ZONE_NAME = 3, "1004", "zone_E"
START_FRAME, END_FRAME = 633, 719

tracks = pd.read_csv(TRACKS_CSV)
with open(ZONE_JSON, "r") as f:
    zone_data = json.load(f)

def find_zone(obj, name):
    if isinstance(obj, dict):
        if obj.get("name") == name or obj.get("zone_name") == name:
            pts = obj.get("points", obj.get("polygon", obj.get("coordinates")))
            if pts is not None:
                return pts
        for value in obj.values():
            result = find_zone(value, name)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for item in obj:
            result = find_zone(item, name)
            if result is not None:
                return result
    return None

raw_polygon = find_zone(zone_data, ZONE_NAME)
if raw_polygon is None:
    raise ValueError(f"Could not find polygon for {ZONE_NAME}.")

polygon = np.array(raw_polygon, dtype=np.int32).reshape(-1, 2)

case = tracks[
    (tracks["view_id"] == VIEW_ID)
    & (tracks["corrected_track_id"].astype(str) == TRACK_ID)
    & tracks["frame_index"].between(START_FRAME, END_FRAME)
]

observed = set(case["frame_index"].astype(int))
missing = sorted(set(range(START_FRAME, END_FRAME + 1)) - observed)
print("V3_1004 | Zone E")
print("Missing frames:", missing)

sample_frames = {633, 644, 655, 666, 677, 688, 700, 719}
for m in missing:
    sample_frames.update(
        range(max(START_FRAME, m - 3), min(END_FRAME, m + 3) + 1)
    )
sample_frames = sorted(sample_frames)

cap = cv2.VideoCapture(VIDEO_PATH)
rows = int(np.ceil(len(sample_frames) / 2))
fig, axes = plt.subplots(rows, 2, figsize=(16, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, frame_no in zip(axes, sample_frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, bgr = cap.read()
    if not ok:
        ax.set_title(f"Frame {frame_no}: unreadable")
        ax.axis("off")
        continue

    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cv2.polylines(image, [polygon], True, (255, 255, 0), 3)

    frame_tracks = tracks[
        (tracks["view_id"] == VIEW_ID)
        & (tracks["frame_index"] == frame_no)
    ]

    target_visible = False
    for _, r in frame_tracks.iterrows():
        x1, y1, x2, y2 = map(
            int, [r["x1"], r["y1"], r["x2"], r["y2"]]
        )
        tid = str(r["corrected_track_id"])
        target = tid == TRACK_ID
        color = (255, 50, 50) if target else (50, 220, 80)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {tid}", (x1, max(22, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
        )
        target_visible |= target

    ax.imshow(image)
    ax.set_title(
        f"Frame {frame_no} | ID 1004: "
        f"{'VISIBLE' if target_visible else 'NOT DETECTED'}"
    )
    ax.axis("off")

for ax in axes[len(sample_frames):]:
    ax.axis("off")

plt.tight_layout()
cap.release()
plt.show()


In [ ]:
import pandas as pd
from pathlib import Path

review_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/"
    "dwell_time_visual_review_queue_v2.csv"
)

review = pd.read_csv(review_path)

mask = (
    (review["view_id"] == 3)
    & (review["zone_name"] == "zone_E")
    & (review["local_person_key"] == "V3_1004")
)

if mask.sum() != 1:
    raise ValueError(
        f"Expected exactly one V3_1004 review row; found {mask.sum()}."
    )

review.loc[mask, "visual_review_status"] = "visually_supported_with_gap"
review.loc[mask, "visual_review_notes"] = (
    "Same physical shopper remains inside Zone E across sampled frames. "
    "One missing detection frame; continuous occupancy during the gap "
    "is not independently confirmed."
)

review.to_csv(review_path, index=False)

print("Saved review status.")
print(
    review.loc[
        mask,
        [
            "view_id",
            "zone_name",
            "local_person_key",
            "duration_sec",
            "missing_frames_in_span",
            "visual_review_status",
            "visual_review_notes",
        ],
    ].to_string(index=False)
)

print("\nReview status counts:")
print(review["visual_review_status"].value_counts().to_string())

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)

visits_path = output_dir / "zone_visits.csv"
review_path = output_dir / "dwell_time_visual_review_queue_v2.csv"

visits = pd.read_csv(visits_path)
review = pd.read_csv(review_path)

print("ORIGINAL ZONE VISITS")
print("Rows:", len(visits))
print("Columns:", visits.columns.tolist())
display(visits.head(5))

print("\nVISUAL REVIEW QUEUE")
print("Rows:", len(review))
print("Columns:", review.columns.tolist())
display(review.head(5))

print("\nREVIEW STATUS COUNTS")
display(
    review["visual_review_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="cases")
)

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)

visits = pd.read_csv(output_dir / "zone_visits.csv")
review = pd.read_csv(
    output_dir / "dwell_time_visual_review_queue_v2.csv"
)

match_cols = [
    "view_id",
    "zone_name",
    "local_person_key",
    "start_frame",
    "end_frame",
]

review_cols = match_cols + [
    "visual_review_status",
    "visual_review_notes",
    "missing_frames_in_span",
]

# Check that the review keys identify individual cases.
if review.duplicated(match_cols).any():
    raise ValueError("Duplicate review keys found. Stop before merging.")

# Attach review information to a separate in-memory copy.
annotated = visits.merge(
    review[review_cols],
    on=match_cols,
    how="left",
    validate="one_to_one",
    indicator=True,
)

matched = annotated["_merge"].eq("both").sum()
unmatched_review = len(review) - matched

print(f"Original visits: {len(visits)}")
print(f"Review cases: {len(review)}")
print(f"Review cases matched: {matched}")
print(f"Review cases not matched: {unmatched_review}")
print(f"Total visits with no manual review: {(annotated['_merge'] == 'left_only').sum()}")

print("\nMatch check:")
display(
    annotated.loc[
        annotated["_merge"].eq("both"),
        match_cols + ["duration_sec", "visual_review_status"],
    ].sort_values(["view_id", "start_frame"])
)

if matched != len(review):
    print("\nWARNING: Some review cases did not match. Do not save final outputs yet.")
else:
    print("\nSUCCESS: Every review case matches an original visit exactly.")


In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)
output_dir.mkdir(parents=True, exist_ok=True)

visits = pd.read_csv(output_dir / "zone_visits.csv")
review = pd.read_csv(
    output_dir / "dwell_time_visual_review_queue_v2.csv"
)

keys = [
    "view_id",
    "zone_name",
    "local_person_key",
    "start_frame",
    "end_frame",
]

review_fields = keys + [
    "visual_review_status",
    "visual_review_notes",
    "missing_frames_in_span",
]

annotated = visits.merge(
    review[review_fields],
    on=keys,
    how="left",
    validate="one_to_one",
)

annotated["review_category"] = annotated["visual_review_status"].fillna(
    "not_manually_reviewed"
)

# Conservative evidence set:
# Exclude visits with unreliable tracking, confirmed ID switches,
# or browsing/identity that remains unconfirmed.
exclude_statuses = {
    "tracking_unreliable",
    "confirmed_local_id_switch",
    "plausible_browsing_unconfirmed",
}

annotated["included_in_conservative_summary"] = (
    ~annotated["review_category"].isin(exclude_statuses)
)

# Preserve the original duration and add explicit evidence labels.
annotated_path = output_dir / "zone_visits_review_annotated_v1.csv"
annotated.to_csv(annotated_path, index=False)

# Summary 1: all original visits, with review categories.
all_summary = (
    annotated.groupby(["view_id", "zone_name"], as_index=False)
    .agg(
        total_visits=("duration_sec", "size"),
        total_observed_span_sec=("duration_sec", "sum"),
        mean_visit_span_sec=("duration_sec", "mean"),
        median_visit_span_sec=("duration_sec", "median"),
        max_visit_span_sec=("duration_sec", "max"),
        manually_reviewed_visits=(
            "visual_review_status",
            lambda s: s.notna().sum(),
        ),
    )
)

all_summary_path = output_dir / "zone_dwell_summary_all_review_labeled_v1.csv"
all_summary.to_csv(all_summary_path, index=False)

# Summary 2: conservative set only.
conservative = annotated[
    annotated["included_in_conservative_summary"]
].copy()

conservative_summary = (
    conservative.groupby(["view_id", "zone_name"], as_index=False)
    .agg(
        included_visits=("duration_sec", "size"),
        total_observed_span_sec=("duration_sec", "sum"),
        mean_visit_span_sec=("duration_sec", "mean"),
        median_visit_span_sec=("duration_sec", "median"),
        max_visit_span_sec=("duration_sec", "max"),
    )
)

conservative_path = (
    output_dir / "zone_dwell_summary_conservative_v1.csv"
)
conservative_summary.to_csv(conservative_path, index=False)

print("Saved files:")
print(annotated_path)
print(all_summary_path)
print(conservative_path)

print("\nReview category counts:")
print(annotated["review_category"].value_counts().to_string())

print("\nOriginal visit count:", len(annotated))
print("Included in conservative summary:", len(conservative))
print("Excluded from conservative summary:", len(annotated) - len(conservative))

print("\nConservative dwell summary:")
display(conservative_summary.sort_values(["view_id", "zone_name"]))

print(
    "\nNote: durations are observed track spans, not proof of uninterrupted "
    "physical presence or confirmed shopper identity."
)

In [ ]:
import pandas as pd
from pathlib import Path

out = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1"
)

for filename in [
    "preliminary_zone_flags.csv",
    "heatmap_summary.csv",
    "view_occupancy_movement_summary.csv",
]:
    path = out / filename
    df = pd.read_csv(path)

    print(f"\n--- {filename} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    display(df.head(10))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild"
)

tracks_path = base / "all_views_tracks_manual_corrections_v5.csv"
tracks = pd.read_csv(tracks_path)

required = {
    "frame_index", "view_id", "corrected_track_id",
    "foot_x", "foot_y"
}
missing = required - set(tracks.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Use corrected IDs when available, falling back to the local track ID.
tracks["person_key"] = tracks["corrected_track_id"].astype(str)
tracks.loc[
    tracks["corrected_track_id"].isna(), "person_key"
] = tracks.loc[
    tracks["corrected_track_id"].isna(), "track_id"
].astype(str) if "track_id" in tracks.columns else "unknown"

# Views are exactly 240 frames each at 24 FPS.
tracks["second_in_view"] = (
    (tracks["frame_index"] % 240) // 24
).astype(int)

tracks["frame_in_view"] = tracks["frame_index"] % 240

# Unique tracked people per frame, then average across frames in each second.
per_frame = (
    tracks.groupby(["view_id", "frame_index"])["person_key"]
    .nunique()
    .rename("people_count")
    .reset_index()
)

per_frame["second_in_view"] = (
    (per_frame["frame_index"] % 240) // 24
).astype(int)

occupancy = (
    per_frame.groupby(["view_id", "second_in_view"])
    .agg(
        mean_people=("people_count", "mean"),
        peak_people=("people_count", "max"),
        frames_counted=("people_count", "size"),
    )
    .reset_index()
)

# Approximate movement between consecutive observed positions for each track.
tracks = tracks.sort_values(["view_id", "person_key", "frame_index"])
tracks["dx"] = tracks.groupby(["view_id", "person_key"])["foot_x"].diff()
tracks["dy"] = tracks.groupby(["view_id", "person_key"])["foot_y"].diff()
tracks["df"] = tracks.groupby(["view_id", "person_key"])["frame_index"].diff()

valid = tracks["df"].gt(0) & tracks["df"].le(5)
tracks["speed_px_sec"] = np.where(
    valid,
    np.hypot(tracks["dx"], tracks["dy"]) * 24 / tracks["df"],
    np.nan,
)
tracks["second_in_view"] = (
    (tracks["frame_index"] % 240) // 24
).astype(int)

movement = (
    tracks.groupby(["view_id", "second_in_view"])["speed_px_sec"]
    .median()
    .rename("median_speed_px_sec")
    .reset_index()
)

second_summary = occupancy.merge(
    movement,
    on=["view_id", "second_in_view"],
    how="left",
)

save_path = base / "vendora_analytics_outputs_v1" / "per_second_occupancy_movement_v1.csv"
second_summary.to_csv(save_path, index=False)

print("Saved:", save_path)
print("Rows:", len(second_summary))
display(
    second_summary.sort_values(
        ["view_id", "second_in_view"]
    ).head(20)
)

In [ ]:
import pandas as pd
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild"
)
path = (
    base / "vendora_analytics_outputs_v1"
    / "per_second_occupancy_movement_v1.csv"
)

df = pd.read_csv(path)

rows = []
for view_id, g in df.groupby("view_id"):
    valid = g.dropna(
        subset=["mean_people", "median_speed_px_sec"]
    )

    # Spearman correlation is descriptive, not proof of causation.
    corr = (
        valid["mean_people"].corr(
            valid["median_speed_px_sec"],
            method="spearman",
        )
        if len(valid) >= 3
        else float("nan")
    )

    rows.append({
        "view_id": int(view_id),
        "mean_occupancy_across_seconds": g["mean_people"].mean(),
        "max_1s_mean_occupancy": g["mean_people"].max(),
        "max_people_in_any_frame": g["peak_people"].max(),
        "median_speed_across_seconds": g["median_speed_px_sec"].median(),
        "min_1s_median_speed": g["median_speed_px_sec"].min(),
        "max_1s_median_speed": g["median_speed_px_sec"].max(),
        "occupancy_speed_spearman": corr,
    })

summary = pd.DataFrame(rows).sort_values("view_id")

out = path.parent / "view_temporal_screening_summary_v1.csv"
summary.to_csv(out, index=False)

print("Saved:", out)
display(summary.round(3))

In [ ]:
import pandas as pd
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild"
)
path = (
    base / "vendora_analytics_outputs_v1"
    / "per_second_occupancy_movement_v1.csv"
)

df = pd.read_csv(path)

priority_views = [1, 2, 5, 6]
subset = df[df["view_id"].isin(priority_views)].copy()

print("LOWEST-SPEED SECONDS")
display(
    subset.sort_values("median_speed_px_sec")
    .groupby("view_id", group_keys=False)
    .head(2)
    .sort_values(["view_id", "second_in_view"])
    .round(3)
)

print("\nHIGHEST-OCCUPANCY SECONDS")
display(
    subset.sort_values("mean_people", ascending=False)
    .groupby("view_id", group_keys=False)
    .head(2)
    .sort_values(["view_id", "second_in_view"])
    .round(3)
)

In [ ]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import math
from pathlib import Path
from IPython.display import display

video_path = "/content/combined_cctv.mp4"

summary_path = Path(
    "/content/drive/MyDrive/vendora_botsort_reid_rebuild/"
    "vendora_analytics_outputs_v1/per_second_occupancy_movement_v1.csv"
)

df = pd.read_csv(summary_path)

# Selected intervals:
# low-speed intervals and high-occupancy intervals from Views 1, 2, 5, 6.
selected = [
    (1, 0), (1, 8),
    (2, 6), (2, 9),
    (5, 4), (5, 5), (5, 6),
    (6, 6), (6, 7), (6, 9),
]

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
frames = []

for view_id, second in selected:
    row = df[
        (df["view_id"] == view_id)
        & (df["second_in_view"] == second)
    ]

    if len(row) != 1:
        print(f"Skipping View {view_id}, second {second}: summary row missing")
        continue

    row = row.iloc[0]

    # Select the middle frame of the chosen second.
    frame_index = (view_id - 1) * 240 + second * 24 + 12

    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
    ok, frame = cap.read()

    if not ok:
        print(f"Could not read frame {frame_index}")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    title = (
        f"View {view_id} | second {second}\n"
        f"Mean people: {row['mean_people']:.1f} | "
        f"Median speed: {row['median_speed_px_sec']:.1f} px/s"
    )
    frames.append((frame, title))

cap.release()

cols = 2
rows = math.ceil(len(frames) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4.8))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax, (frame, title) in zip(axes, frames):
    ax.imshow(frame)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

for ax in axes[len(frames):]:
    ax.axis("off")

plt.tight_layout()
plt.show()
